<a href="https://colab.research.google.com/github/AngeP02/MonteCarloPerRadioterapia/blob/main/Progetto_HPC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup ambiente di sviluppo

In [ ]:
!nvidia-smi                    # che GPU hai
!nvcc --version                # versione CUDA
!gcc --version                 # compilatore C++
!python3 --version             # Python per i grafici
!pip install numpy matplotlib  # librerie plotting

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Python 3.12.13


Creazione della struttura delle celle

In [ ]:
import os
os.makedirs('mc_rt_cpu', exist_ok=True)
print("Cartella creata")

# Codice CPU Sequenziale

## Implementazione

### File compton.h

Effetto Compton comprende:
1.   Metodo di Kahn che campiona l'angolo di scattering Compton dalla distribuzione di Klein-Nishina attraverso il rejection sampling.
      * Input
        * energy_mev: energia fotone incidente MeV
        * xi_1, xi_2, xi_3: numeri casuali
      * Output
        * cos_theta: cosenzo angolo di scattering
        * energia_scatter: energia fotone diffuso

      Si hanno due fasi:
        * decomposizione in cui viene divisa la sezione d'urto in due rami probabilistici e si sceglie un ramo in base al numero casuale e si genera un valore di tau;
        * criterio di accettazione, in cui questo valore tau viene accettato solo se supera un test statistico basato sulla conservazione del momento e dell'energia. Se il test non viene superato si ricomincia.

2.   Wrapper con loop di rejection che garantisce che la funzione non restituisca mai un valore fisicamente impossivile.
3.   Rotazione della direzione del fotone dopo lo scattering. In input prende il versore attuale, viene calcolato un nuovo angolo e applicata una matrice di rotazione così da ottnere un nuovo versore di direzione che il fotone seguirà nel voxel successivo.

        * Input
            * ux, uy, uz che sono i versori e rappresentano la direzione del fotone prima dell'urto.



In [2]:
%%writefile compton.h
#pragma once

#include <cmath>
#include "physics.h"

// Kahn
inline void kahn_compton(double energia_mev, double xi_1, double xi_2, double xi_3, double &cos_theta, double &energia_scatter) {
    double alpha = energia_mev / ME_C2;
    double tau_min = 1.0 / (1.0 + 2.0 * alpha);

    double area_ramo_1  = std::log(1.0 / tau_min);
    double area_ramo_2  = (1.0 - tau_min * tau_min) * 0.5;
    double area_totale = area_ramo_1 + area_ramo_2;
    double tau;

    if (xi_1 * area_totale < area_ramo_1) {
        // si campiona ramo logaritmico
        tau = std::pow(tau_min, 1.0 - xi_2);
    } else {
        // si fa interpolazione lineare e si campiona ramo lineare
        double tau_min_quadro = tau_min * tau_min;
        double tau_quadro = tau_min_quadro + xi_2*(1.0 - tau_min_quadro);
        tau = std::sqrt(std::max(tau_quadro, 1e-30));
    }

    tau = std::min(std::max(tau, tau_min), 1.0);

    cos_theta = 1.0 - (1.0 - tau) / (alpha * tau);
    cos_theta = std::min(std::max(cos_theta, -1.0), 1.0);
    energia_scatter = tau * energia_mev;

    // calcolo della probabilità di accettazione
    double sin2_theta = std::max(0.0, 1.0 - cos_theta * cos_theta);
    double termine_correttivo = (tau * sin2_theta) / (1.0 + tau * tau);
    double probabilita_accettazione = 1.0 - termine_correttivo;
    probabilita_accettazione = std::max(0.0, std::min(probabilita_accettazione, 1.0));

    if (xi_3 > probabilita_accettazione) {
        cos_theta = 2.0;
    }
}

// Wrapper con loop rejection
template<typename RNG>
inline void sample_compton(double energia_mev, RNG &rng, double &cos_theta, double &energia_scatter) {
    while(true){ // loop rejection perchè prima o poi un valore valido viene estratto
        double xi_1 = rng();
        double xi_2 = rng();
        double xi_3 = rng();
        kahn_compton(energia_mev, xi_1, xi_2, xi_3, cos_theta, energia_scatter);
        if (cos_theta <= 1.0)
            return;
    }
}

// Rotazione della direzione del fotone dopo lo scattering
inline void rotate_direction(double &ux, double &uy, double &uz, double cos_theta, double phi) {
    double sin_theta = std::sqrt(std::max(0.0, 1.0 - cos_theta * cos_theta));
    double cos_phi = std::cos(phi);
    double sen_phi = std::sin(phi);

    double ux_new, uy_new, uz_new;

    if (std::fabs(uz) > 0.99999) { // se viaggia quasi parallelo all'asse Z
        double segno = 1.0;
        if(uz > 0.0){
          segno = 1.0;
        }else{
          segno = -1.0;
        }
        ux_new = sin_theta * cos_phi;
        uy_new = sin_theta * sen_phi * segno;
        uz_new = cos_theta * segno;
    } else {
        double proiezione_xy = std::sqrt(1.0 - uz * uz);
        ux_new = sin_theta * (ux * uz * cos_phi - uy * sen_phi) / proiezione_xy + ux * cos_theta;
        uy_new = sin_theta * (uy * uz * cos_phi + ux * sen_phi) / proiezione_xy + uy * cos_theta;
        uz_new = -sin_theta * cos_phi * proiezione_xy + uz * cos_theta;
    }

    // Normalizzazione
    double norm = std::sqrt(ux_new * ux_new + uy_new * uy_new + uz_new * uz_new);
    if (norm > 0.0){
       ux = ux_new/norm;
       uy = uy_new/norm;
       uz = uz_new/norm;
    }
}

Writing compton.h


### File phantom.h

Si occupa della costruzione del phantom voxelizzato. In particolare si hanno:
* Phantom Omogeneo
  * Si considera composto da acqua pura di 30x30x30 cm^3, quindi 100^3 voxel con 3mm a lato
* Phantom acqua + inserto osseo
  * si considera composto oltre che da acqua da un inserto osseo di 5x5x5 cm^3 al centro

Inizializza un volume cubico stadard dove ogni singolo voxel è marcato con acqua e a quello con inserto osseo viene inserito al centro della griglia un cubo di materiale osseo. *Viene, inoltre, considerato che metà lato dell'inserto è pari a 2.5cm / 0.3cm/voxel che è circa uguale e 8 voxel*

In [3]:
%%writefile phantom.h

#pragma once

#include <cstring>
#include <cstdio>
#include "physics.h"

// Phantom Omogeneo (solo acqua)
inline void build_phantom_water(int *phantom) {
    int total = NX * NY * NZ;
    for (int i = 0; i < total; i++)
        phantom[i] = MAT_WATER;
}

// Phantom acqua + inserto osseo
inline void build_phantom_hetero(int *phantom) {
    build_phantom_water(phantom);

    // Centro del phantom in indici voxel
    int cx = NX / 2;
    int cy = NY / 2;
    int cz = NZ / 2;

    int meta_lato_inserto = (int)(2.5 / VOXEL_CM + 0.5);

    int count = 0;
    for (int iz = cz - meta_lato_inserto; iz < cz + meta_lato_inserto; iz++)
    for (int iy = cy - meta_lato_inserto; iy < cy + meta_lato_inserto; iy++)
    for (int ix = cx - meta_lato_inserto; ix < cx + meta_lato_inserto; ix++) {
        if (ix >= 0 && ix < NX && iy >= 0 && iy < NY && iz >= 0 && iz < NZ) { // controlli per non scrivere fuori dalla memoria del phantom
            phantom[phantom_idx(ix, iy, iz)] = MAT_BONE;
            count++;
        }
    }

    double volume_fisico_totale = count * VOXEL_CM * VOXEL_CM * VOXEL_CM;
    printf("Inserto osseo: %d voxel = %.1f cm³  ( volume teorico 125 cm³)\n", count, volume_fisico_totale);
}

// Inizializzazione dose grid a zero
inline void init_dose(double *dose) {
    memset(dose, 0, NX * NY * NZ * sizeof(double));
}


Writing phantom.h


### File physics.h

Contiene le costanti fondamentali al calcolo e le tabelle dei coefficienti di interazione derivate dal database ufficiale NIST XCOM (https://physics.nist.gov/PhysRefData/XrayMassCoef/tab4.html), oltre alle funzioni matematiche per l'estrazione delle probabilità fisiche. Vengono definite le soglie ECUT e PCUT, la risoluzione spaziale della griglia voxelizzata e la cnversione fisica tra idnici dei voxel.

La griglia energetica viene considerata come l'insieme dei punti discreti su cui sono stati misurati e tabulati i dati fisici e secondo il db NIST XCOM si considera da 28 punti, quindi da 0.01 MeV a 20 MeV. In questo modo se il fotone ha un'energia che cade su uno di questi punti viene letto direttamente il valore dalla tabella altrimenti si usa la griglia per sapere tra quali punti effettuare l'interpolazione.

In [4]:
%%writefile physics.h

#pragma once

#include <cmath>
#include <cassert>

// COSTANTI FISIHCHE
static const double ME_C2    = 0.51099895;  // MeV  massa a riposo elettrone
static const double PI       = 3.14159265358979323846;
static const double ECUT     = 0.010;       // MeV  cutoff fotoni  (10 keV)
static const double PCUT     = 0.100;       // MeV  cutoff elettroni

// GEOMETRIA PHANTOM
static const int    NX = 100, NY = 100, NZ = 100;   // voxel per asse
static const double VOXEL_CM = 0.30;                // lato voxel [cm] = 3 mm
static const double PHANTOM_CM = NX * VOXEL_CM;     // 30.0 cm per asse

// INDICI MATERIALI
#define MAT_WATER 0   // acqua  ρ = 1.000 g/cm^3
#define MAT_BONE  1   // osso (ICRU)  ρ = 1.850 g/cm^3
#define MAT_LUNG  2   // polmone (ICRU)  ρ = 0.260 g/cm^3
#define MAT_AIR   3   // aria  ρ = 0.001205 g/cm^3
#define N_MAT     4   // numero materiali disponibili

// DENSITÀ [g/cm^3]
static const double DENSITY[N_MAT] = { 1.000, 1.850, 0.260, 0.001205 };

// GRIGLIA ENERGETICA [MeV]  (28 punti, da 0.01 a 20 MeV)
static const int N_ENERGY = 28;
static const double ENERGY_GRID[N_ENERGY] = {
    0.010, 0.015, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080, 0.100,
    0.150, 0.200, 0.300, 0.400, 0.500, 0.600, 0.800, 1.000, 1.250,
    1.500, 2.000, 3.000, 4.000, 5.000, 6.000, 8.000, 10.000,
    15.000, 20.000
};

// COEFFICIENTI μ/ρ [cm^2/g]  per ogni materiale e processo -> [materiale][bin_energia]
static const double MU_TOTAL[N_MAT][N_ENERGY] = {
    // ACQUA
    { 5.329, 1.673, 0.8096, 0.3756, 0.2683, 0.2269, 0.2059, 0.1837, 0.1707,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01941, 0.01813 },
    // OSSO
    { 19.89, 7.131, 3.085, 1.012, 0.5475, 0.3941, 0.3178, 0.2595, 0.2368,
      0.1958, 0.1698, 0.1393, 0.1222, 0.1107, 0.1018, 0.08795, 0.07838, 0.06945,
      0.06283, 0.05351, 0.04257, 0.03624, 0.03209, 0.02913, 0.02536, 0.02296,
      0.01978, 0.01832 },
    // POLMONE
    { 5.208, 1.638, 0.7933, 0.3681, 0.2629, 0.2224, 0.2018, 0.1800, 0.1673,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01902, 0.01776 },
    // ARIA
    { 36.01, 12.28, 5.279, 1.625, 0.6918, 0.3954, 0.2885, 0.2085, 0.1875,
      0.1504, 0.1340, 0.1123, 0.09921, 0.09076, 0.08414, 0.07285, 0.06529, 0.05817,
      0.05298, 0.04534, 0.03597, 0.03054, 0.02699, 0.02453, 0.02131, 0.01931,
      0.01673, 0.01551 }
};

// EFFETTO FOTOELETTRICO
static const double MU_PE[N_MAT][N_ENERGY] = {
    // ACQUA
    { 4.944, 1.374, 0.5195, 0.1036, 0.02407, 0.005800, 0.001334, 5.510e-5, 3.998e-5,
      2.799e-6, 2.200e-7, 1.400e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // OSSO
    { 19.35, 6.833, 2.818, 0.7469, 0.2837, 0.1152, 0.04660, 0.008680, 0.001900,
      1.800e-4, 2.000e-5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // POLMONE
    { 4.845, 1.346, 0.5091, 0.1015, 0.02359, 0.005684, 0.001307, 5.400e-5, 3.918e-5,
      2.743e-6, 2.156e-7, 1.372e-9, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 },
    // ARIA
    { 35.52, 11.99, 5.012, 1.379, 0.4529, 0.1581, 0.05757, 0.008251, 0.001581,
      8.208e-5, 7.636e-6, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 }
};

// SCATTERING COMPTON
static const double MU_COMPTON[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.3854, 0.2988, 0.2672, 0.2651, 0.2595, 0.2476, 0.2329, 0.1984, 0.1661,
      0.1505, 0.1370, 0.1186, 0.1061, 0.09687, 0.09007, 0.07865, 0.07072, 0.06323,
      0.05754, 0.04942, 0.03969, 0.03403, 0.03031, 0.02770, 0.02429, 0.02219,
      0.01878, 0.01719 },
    // OSSO
    { 0.4869, 0.2684, 0.2503, 0.2465, 0.2429, 0.2310, 0.2172, 0.1848, 0.1548,
      0.1400, 0.1275, 0.1103, 0.09870, 0.09010, 0.08377, 0.07313, 0.06575, 0.05862,
      0.05338, 0.04579, 0.03667, 0.03133, 0.02784, 0.02539, 0.02217, 0.02016,
      0.01702, 0.01552 },
    // POLMONE
    { 0.3777, 0.2928, 0.2619, 0.2598, 0.2543, 0.2426, 0.2282, 0.1944, 0.1628,
      0.1475, 0.1342, 0.1162, 0.1040, 0.09493, 0.08827, 0.07708, 0.06930, 0.06196,
      0.05639, 0.04843, 0.03889, 0.03335, 0.02970, 0.02715, 0.02380, 0.02175,
      0.01840, 0.01684 },
    // ARIA
    { 0.3779, 0.2933, 0.2624, 0.2602, 0.2547, 0.2430, 0.2285, 0.1946, 0.1630,
      0.1477, 0.1344, 0.1163, 0.1041, 0.09516, 0.08844, 0.07723, 0.06942, 0.06207,
      0.05649, 0.04852, 0.03894, 0.03339, 0.02973, 0.02718, 0.02382, 0.02177,
      0.01843, 0.01686 }
};

// PRODUZIONE DI COPPIE
static const double MU_PAIR[N_MAT][N_ENERGY] = {
    // ACQUA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000630, 0.000940 },
    // OSSO
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.002760, 0.002800 },
    // POLMONE
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000617, 0.000921 },
    // ARIA
    { 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
      0.000589, 0.000879 }
};

// INTERPOLAZIONE LINEARE SU GRIGLIA ENERGETICA
inline double interp_mu(double energia_mev, const double tabella[N_ENERGY]) {
    if (energia_mev <= ENERGY_GRID[0])
        return tabella[0];
    if (energia_mev >= ENERGY_GRID[N_ENERGY-1])
        return tabella[N_ENERGY-1];

    int indice_limite_inferiore = 0;
    int indice_imite_superiore = N_ENERGY - 1;

    while (indice_imite_superiore - indice_limite_inferiore > 1) {
        int punto_centrale = (indice_limite_inferiore + indice_imite_superiore) / 2;
        if (ENERGY_GRID[punto_centrale] <= energia_mev){
            indice_limite_inferiore = punto_centrale;
        }else{
            indice_imite_superiore = punto_centrale;
        }
    }

    double fattore_interpolazione = (energia_mev - ENERGY_GRID[indice_limite_inferiore]) / (ENERGY_GRID[indice_imite_superiore] - ENERGY_GRID[indice_limite_inferiore]);
    return tabella[indice_limite_inferiore] * (1.0 - fattore_interpolazione) + tabella[indice_imite_superiore] * fattore_interpolazione;
}

// CALCOLO MU TOTALE
inline double get_mu_total(double energia, int materiale) {   // probabilita di avere un urto
    return interp_mu(energia, MU_TOTAL[materiale]) * DENSITY[materiale];
}
inline double get_mu_pe(double energia, int materiale) {      // probabilita effetto fotoelettrico
    return interp_mu(energia, MU_PE[materiale]) * DENSITY[materiale];
}
inline double get_mu_compton(double energia, int materiale) {   // probabilità di compton
    return interp_mu(energia, MU_COMPTON[materiale]) * DENSITY[materiale];
}
inline double get_mu_pair(double energia, int materiale) {    // probabilita produzione coppie
    return interp_mu(energia, MU_PAIR[materiale]) * DENSITY[materiale];
}

// SELEZIONE TIPO DI INTERAZIONE
// Restituisce: 0=fotoelettrico, 1=Compton, 2=produzione coppie
// xi: numero casuale uniforme in [0,1)
inline int select_interaction(double energia, int materiale, double xi) {
    double mu_totale = get_mu_total(energia, materiale);

    if (mu_totale <= 0.0)
        return 1;

    double probabilita_fotoelettrico = get_mu_pe(energia, materiale) / mu_totale;
    double probabilita_compton = get_mu_compton(energia, materiale) / mu_totale;

    if (xi < probabilita_fotoelettrico)
        return 0;   // fotoelettrico
    if (xi < probabilita_fotoelettrico + probabilita_compton)
        return 1; // compton
    return 2;  // produzione di coppie
}

// INDICE LINEARE PHANTOM CON ROW MAJOR ORDER
inline int phantom_idx(int ix, int iy, int iz) {
    return ix + NX * (iy + NY * iz);
}

// CONTROLLO COORDINATE CONTORNO CUBO
inline bool inside(double x, double y, double z) {
    return x >= 0.0 && x < PHANTOM_CM &&
           y >= 0.0 && y < PHANTOM_CM &&
           z >= 0.0 && z < PHANTOM_CM;
}

// CONTROLLO PSIZIONE IN VOXEL
inline int vox(double coord) {
    int indice_voxel = (int)(coord / VOXEL_CM);
    if (indice_voxel < 0)
        indice_voxel = 0;
    if (indice_voxel >= NX)
        indice_voxel = NX - 1;
    return indice_voxel;
}


Writing physics.h


### File random.h

Si occupa di generare numeri casuali ad alte prestazioni e campionare l'energia dei fotoni incidenti seguendo lo spettro reale di un acceleratore lineare LINAC. In particolare:
* è stato implementato l'algoritmo xoshiro256
  * viene utilizzato l'algoritmo splotmix64 per garantire che partendo da un singolo seed tutti i bit sono inizializzati in modo correlato
* è stato modellato un fascio reale di radioterapia (basato sui dati della lettetatura) e questo spettro è stato suddiviso in 24 intervalli energetici (bin) da 0.25 MeV fino a 6 MeV.
  * Si fa riferimento allo spettro 6MV del fascio standard Varian 2100C a 6MV, campo 10x10cm^2 a SAD 100cm

In [5]:
%%writefile random.h

#pragma once

#include <cstdint>
#include <cmath>
#include <cstring>

// XOSHIRO256
struct Xoshiro256 {
    uint64_t s[4];

    // Inizializzazione con un seed a 64 bit usando splitmix64
    explicit Xoshiro256(uint64_t seed) { // con explicit si evitano conversioni automatiche
        auto splitmix = [](uint64_t &x) -> uint64_t {
            x += 0x9e3779b97f4a7c15ULL; // rappresentazione a 64 bit della sezione aurea (2^64/phi)
            uint64_t z = x;
            z = (z ^ (z >> 30)) * 0xbf58476d1ce4e5b9ULL;
            z = (z ^ (z >> 27)) * 0x94d049bb133111ebULL;
            return z ^ (z >> 31);
        };
        s[0] = splitmix(seed);
        s[1] = splitmix(seed);
        s[2] = splitmix(seed);
        s[3] = splitmix(seed);
    }

    // Genera un uint64_t casuale
    uint64_t next() {
        const uint64_t result = rotate_left(s[1] * 5, 7) * 9;
        const uint64_t t = s[1] << 17;
        s[2] ^= s[0]; s[3] ^= s[1]; // ^ indica opertaroe XOR bit a bit
        s[1] ^= s[2]; s[0] ^= s[3];
        s[2] ^= t;
        s[3] = rotate_left(s[3], 45);
        return result;
    }

    // Genera double uniforme in (0, 1) escludendo 0 per evitare log(0)
    double operator()() {
        double r;
        do {
            // usa i 53 bit superiori
            r = (double)(next() >> 11) * (1.0 / (double)(1ULL << 53)); // ULL intero a 64 bit senza segno
        } while (r <= 0.0);
        return r;
    }

private:
    static uint64_t rotate_left(const uint64_t x, int k) {
        return (x << k) | (x >> (64 - k));
    }
};

// SPETTRO 6MV
static const int SPECTRUM_BINS = 24;
static const double SPECTRUM_E[SPECTRUM_BINS] = {
    0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 1.75, 2.00,
    2.25, 2.50, 2.75, 3.00, 3.25, 3.50, 3.75, 4.00,
    4.25, 4.50, 4.75, 5.00, 5.25, 5.50, 5.75, 6.00
};

// Fluenza relativa normalizzata  (somma = 1.0)
static const double SPECTRUM_FLUENCE[SPECTRUM_BINS] = {
    0.0243, 0.0676, 0.0862, 0.0929, 0.0919, 0.0868, 0.0794, 0.0712,
    0.0628, 0.0548, 0.0471, 0.0399, 0.0334, 0.0276, 0.0224, 0.0178,
    0.0138, 0.0104, 0.0075, 0.0052, 0.0034, 0.0020, 0.0010, 0.0004
};

// CDF precalcolata all'inizializzazione
struct Spectrum {
    double cdf[SPECTRUM_BINS];
    double energie[SPECTRUM_BINS];
    double larghezza_intervallo_energetico_bin;

    Spectrum() {
        double somma_fluence = 0.0;
        for (int i = 0; i < SPECTRUM_BINS; i++) {
          somma_fluence += SPECTRUM_FLUENCE[i];
        }

        cdf[0] = SPECTRUM_FLUENCE[0] / somma_fluence;
        for (int i = 1; i < SPECTRUM_BINS; i++)
            cdf[i] = cdf[i-1] + SPECTRUM_FLUENCE[i] / somma_fluence;
        cdf[SPECTRUM_BINS-1] = 1.0;

        for (int i = 0; i < SPECTRUM_BINS; i++)
            energie[i] = SPECTRUM_E[i];

        larghezza_intervallo_energetico_bin = 0.25;
    }

    // Campiona energia con binary search sulla CDF
    double sample(Xoshiro256 &rng) const {
        double xi = rng();
        // Ricerca binaria sulla CDF
        int indice_limite_inferiore = 0;
        int indice_limite_superiore = SPECTRUM_BINS - 1;
        while (indice_limite_inferiore < indice_limite_superiore) {
            int punto_centrale = (indice_limite_inferiore + indice_limite_superiore) / 2;
            if (cdf[punto_centrale] < xi){
                indice_limite_inferiore = punto_centrale + 1;
            }
            else{
                indice_limite_superiore = punto_centrale;
            }
        }

        double energia_centrale = energie[indice_limite_inferiore];
        double offset = (rng() - 0.5) * larghezza_intervallo_energetico_bin;
        double energia = energia_centrale + offset;

        if (energia < 0.01){
           energia = 0.01;
        }
        if (energia > 6.00){
          energia = 6.00;
        }
        return energia;
    }
};


Writing random.h


### File output.h

Contiene le routine necessarie per elaborare la matrice tridimensionale della dose e generare grafici e statistiche. In particolare:
* PDD o Percent Depht Dose è la misura di come la dose varia man mano che il fascio penetra in profondità nel paziente. Considera:
  * Media spaziale che calcola la media su una finestra di +- 8 voxel per simulare la risposta di una camera a ionizzazione reale;
  * normalizzazione che imposta la dose massima lungo l'asse mentre le altre vengono scalate
  * costruzione di profondità che calcola la coordinata fisica per ogni punto di misura.

* Calcola la funzione che analizza come la dose si distribuisce trasversalmente rispetto alla direzione del fascio
* Esportazione dei dati in csv
* Monitoraggio efficienza
  * numero di particelle al secondo;
  * energia depositata
  * voxel occupati

In [6]:
%%writefile output.h

#pragma once

#include <cstdio>
#include <cmath>
#include <fstream>
#include <algorithm>
#include "physics.h"

// PDD
inline void compute_pdd(const double *dose, double *pdd, double *profondita, int semiampiezza_della_media = 8) {
    int cx = NX / 2; // centro del fascio in X
    int cy = NY / 2; // centro del fascio in Y

    double max_dose = 0.0;
    for (int iz = 0; iz < NZ; iz++) {
        double valore_dose = 0.0;
        int    num_voxel_sommati = 0;
        for (int ix = cx - semiampiezza_della_media; ix <= cx + semiampiezza_della_media; ix++){
          for (int iy = cy - semiampiezza_della_media; iy <= cy + semiampiezza_della_media; iy++) {
              if (ix >= 0 && ix < NX && iy >= 0 && iy < NY) {
                  valore_dose += dose[phantom_idx(ix, iy, iz)];
                  num_voxel_sommati++;
              }
          }
        }
        if (num_voxel_sommati > 0) {
            pdd[iz] = valore_dose / num_voxel_sommati;
        } else{
            0.0;
        }
        profondita[iz] = (iz + 0.5) * VOXEL_CM;
        if (pdd[iz] > max_dose){
            max_dose = pdd[iz];
        }
    }

    if (max_dose > 0.0)
        for (int iz = 0; iz < NZ; iz++)
            pdd[iz] = pdd[iz] / max_dose * 100.0;
}

// PROFILO LATERALE a profondità fissa (lungo X, centrato su Y)
inline void compute_lateral_profile(const double *dose, double *profilo_dose_relativa, double *coordinate_cm, double profondita_scelta = 10.0, int semiampiezza_media = 2) {
    int iz = std::min((int)(profondita_scelta / VOXEL_CM), NZ - 1);
    int centro_asse_y = NY / 2;

    double dose_massima_profilo = 0.0;
    for (int ix = 0; ix < NX; ix++) {
        double somma_dose_accumulata = 0.0;
        int conteggio_voxel_validi = 0;
        for (int iy = centro_asse_y - semiampiezza_media; iy <= centro_asse_y + semiampiezza_media; iy++) {
            if (iy >= 0 && iy < NY) {
                somma_dose_accumulata += dose[phantom_idx(ix, iy, iz)];
                conteggio_voxel_validi++;
            }
        }
        if (conteggio_voxel_validi > 0){
            profilo_dose_relativa[ix] = somma_dose_accumulata / conteggio_voxel_validi;
        } else{
            profilo_dose_relativa[ix] = 0.0;
        }
        coordinate_cm[ix] = (ix + 0.5) * VOXEL_CM - PHANTOM_CM / 2.0;
        if (profilo_dose_relativa[ix] > dose_massima_profilo) {
            dose_massima_profilo = profilo_dose_relativa[ix];
        }
    }

    if (dose_massima_profilo > 0.0)
        for (int ix = 0; ix < NX; ix++)
            profilo_dose_relativa[ix] = profilo_dose_relativa[ix] / dose_massima_profilo * 100.0;
}

// SALVA PDD SU CSV
inline void save_pdd_csv(const double *vettore_profondita_cm, const double *pdd, const char *filename) {
    std::ofstream f(filename);
    f << "depth_cm,dose_percent\n";
    for (int iz = 0; iz < NZ; iz++)
        f << vettore_profondita_cm[iz] << "," << pdd[iz] << "\n";
    f.close();
    printf("Salvato: %s\n", filename);
}

// SALVA PROFILO LATERALE SU CSV
inline void save_profile_csv(const double *coordinate_cm, const double *profilo_dose_relativa, const char *filename) {
    std::ofstream f(filename);
    f << "position_cm,dose_percent\n";
    for (int ix = 0; ix < NX; ix++)
        f << coordinate_cm[ix] << "," << profilo_dose_relativa[ix] << "\n";
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA SLICE 2D CENTRALE SU CSV (per heatmap)
inline void save_dose_slice_csv(const double *dose, const char *filename) {
    std::ofstream f(filename);
    int iy = NY / 2;
    for (int iz = 0; iz < NZ; iz++) {
        for (int ix = 0; ix < NX; ix++) {
            f << dose[phantom_idx(ix, iy, iz)];
            if (ix < NX - 1) f << ",";
        }
        f << "\n";
    }
    f.close();
    printf("  Salvato: %s\n", filename);
}

// SALVA DOSE 3D COMPLETA
inline void save_dose_binary(const double *dose, const char *filename) {
    FILE *f = fopen(filename, "wb");
    if (!f) { printf("ERRORE: impossibile aprire %s\n", filename); return; }
    fwrite(dose, sizeof(double), NX * NY * NZ, f);
    fclose(f);
    printf("Salvato: %s  (%d float64)\n", filename, NX * NY * NZ);
}

// STAMPA TABELLA PDD AI PUNTI DI RIFERIMENTO CLINICI
inline void print_pdd_table(const double *profondita, const double *pdd, const char *label) {
    printf("\n  PDD — %s\n", label);
    printf("  %-20s  %10s  %s\n", "Profondità [cm]", "Dose [%]", "Riferimento");
    printf("  %s\n", "─────────────────────────────────────────");

    double refs[]      = {1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0};
    const char *notes[] = {"build-up", "d_max 6MV", "", "D10", "", "D20", ""};

    for (int r = 0; r < 7; r++) {
        int k = (int)(refs[r] / VOXEL_CM);
        if (k >= 0 && k < NZ)
            printf("  %-20.1f  %10.2f  %s\n", profondita[k], pdd[k], notes[r]);
    }
}

// STATISTICHE GENERALI SULLA DOSE
inline void print_dose_stats(const double *dose, long long numero_particelle_totali, double tempo_esecuzione_secondi) {
    double dose_massima_assoluta = 0.0;
    double energia_totale_depositata = 0.0;
    int conteggio_voxel_colpiti = 0;

    for (int i = 0; i < NX * NY * NZ; i++) {
        if (dose[i] > 0.0) {
            conteggio_voxel_colpiti++;
            energia_totale_depositata += dose[i];
            if (dose[i] > dose_massima_assoluta){
                dose_massima_assoluta = dose[i];
            }
        }
    }

    printf("\n Statistiche simulazione: \n");
    printf("  Particelle simulate : %lld\n",  numero_particelle_totali);
    printf("  Tempo totale        : %.2f s\n", tempo_esecuzione_secondi);
    printf("  Throughput          : %.3f MP/s\n", numero_particelle_totali / tempo_esecuzione_secondi / 1.0e6);
    printf("  Tempo/particella    : %.1f μs\n", tempo_esecuzione_secondi / numero_particelle_totali * 1.0e6);
    printf("  Voxel con dose>0    : %d / %d (%.1f%%)\n", conteggio_voxel_colpiti, NX*NY*NZ, 100.0*conteggio_voxel_colpiti/(NX*NY*NZ));
    printf("  Energia totale dep. : %.4e MeV\n", energia_totale_depositata);
    printf("  Energia/particella  : %.4e MeV\n", numero_particelle_totali > 0 ? energia_totale_depositata / numero_particelle_totali : 0.0);
    printf("  Dose massima (u.a.) : %.6e\n", dose_massima_assoluta);
}


Writing output.h


### File main.cpp

Programma principale che unisce tutti i moduli ed esegue la simulazione Monte Carlo per radioterapia. Effettua:
* Campionamento della sorgente;
* Applica algoritmo di Traversal;
* Gestisce la storia di ogni fotone

Implementa il trasporto di fotoni in un phantom voxelizzato con:
 * Spettro 6MV (Sheikh-Bagheri & Rogers 2002)
 * Legge di Beer-Lambert + voxel traversal (Amanatides & Woo 1987)
 * Effetto fotoelettrico, Compton (metodo di Kahn), produzione coppie
 * Sezioni d'urto da NIST XCOM (Hubbell & Seltzer 1996)
 * Approssimazione KERMA ≈ dose (deposito locale elettroni)
 * PRNG: xoshiro256** (Blackman & Vigna 2018)

In [7]:
%%writefile main.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// stato di una particella sullo stack
struct Particle {
    double x, y, z;    // posizione [cm]
    double ux, uy, uz; // versore direzione (normalizzato)
    double energia;     // energia [MeV]
};

// CAMPIONAMENTO SORGENTE
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;

    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0;
    p.uy = 0.0;
    p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// DISTANZA AL PROSSIMO CONFINE DI VOXEL
inline double boundary_distance(double x, double y, double z, double ux, double uy, double uz, int ix, int iy, int iz) {
    double distanza_minima_confine = 1.0e30; // inizializzata a infinito
    if (std::fabs(ux) > 1.0e-12) {
        double confine_voxel_X;
        if (ux > 0){
            confine_voxel_X = (ix + 1) * VOXEL_CM;
        } else{
            confine_voxel_X = ix * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_X - x) / ux;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uy) > 1.0e-12) {
        double confine_voxel_Y;
        if (uy > 0){
            confine_voxel_Y = (iy + 1) * VOXEL_CM;
        } else{
            confine_voxel_Y = iy * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Y - y) / uy;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uz) > 1.0e-12) {
        double confine_voxel_Z;
        if (uz > 0){
            confine_voxel_Z = (iz + 1) * VOXEL_CM;
        } else{
            confine_voxel_Z = iz * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Z - z) / uz;
        if (distanza_lineare > 1.0e-10) {
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    return distanza_minima_confine;
}

// TRASPORTO FOTONE CICLO COMPLETO
void transport_photon(Particle fotone_iniziale, const int *phantom, double *dose, Xoshiro256 &rng) {

    Particle stack[64];
    int num_particelle_stack = 0;
    stack[num_particelle_stack++] = fotone_iniziale;

    while (num_particelle_stack > 0) {
        Particle particella_corrente = stack[--num_particelle_stack];
        for (int step = 0; step < 100000; step++) {
            // Cutoff energetico
            if (particella_corrente.energia < ECUT) {
                // Deposita energia residua nel voxel corrente
                if (inside(particella_corrente.x, particella_corrente.y, particella_corrente.z)) {
                    int ix = vox(particella_corrente.x);
                    int iy = vox(particella_corrente.y);
                    int iz = vox(particella_corrente.z);
                    dose[phantom_idx(ix, iy, iz)] += particella_corrente.energia;
                }
                break;
            }
            // Verifica bounds
            if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                break;
            int ix = vox(particella_corrente.x);
            int iy = vox(particella_corrente.y);
            int iz = vox(particella_corrente.z);
            int materiale = phantom[phantom_idx(ix, iy, iz)];
            double mu = get_mu_total(particella_corrente.energia, materiale); // coefficiente di attenuazione totale

            if (mu <= 0.0)
                break;
            // Campiona cammino libero medio
            double xi = rng();
            double distanza_teorica = -std::log(xi) / mu;
            double distanza_fisica = boundary_distance(particella_corrente.x, particella_corrente.y, particella_corrente.z, particella_corrente.ux, particella_corrente.uy, particella_corrente.uz, ix, iy, iz);

            if (distanza_teorica <= distanza_fisica) {
                // Sposta la particella al punto di interazione
                particella_corrente.x += particella_corrente.ux * distanza_teorica;
                particella_corrente.y += particella_corrente.uy * distanza_teorica;
                particella_corrente.z += particella_corrente.uz * distanza_teorica;

                if (!inside(particella_corrente.x, particella_corrente.y, particella_corrente.z))
                    break;

                // Ricalcola voxel
                ix = vox(particella_corrente.x);
                iy = vox(particella_corrente.y);
                iz = vox(particella_corrente.z);
                materiale = phantom[phantom_idx(ix, iy, iz)];
                int id_voxel = phantom_idx(ix, iy, iz);
                int tipo_interazione = select_interaction(particella_corrente.energia, materiale, rng());

                // FOTOELETTRICO: assorbimento totale
                if (tipo_interazione == 0) {
                    dose[id_voxel] += particella_corrente.energia;
                    break;
                }
                // COMPTON: metodo di Kahn
                else if (tipo_interazione == 1) {
                    double cos_theta;
                    double energia_scatter;
                    sample_compton(particella_corrente.energia, rng, cos_theta, energia_scatter);

                    // Deposita energia ceduta all'elettrone (KERMA locale)
                    double energia_ceduta = particella_corrente.energia - energia_scatter;
                    if (energia_ceduta > 0.0) {
                        dose[id_voxel] += energia_ceduta;
                    }

                    // Aggiorna energia e direzione del fotone
                    particella_corrente.energia = energia_scatter;
                    double phi = 2.0 * PI * rng();
                    rotate_direction(particella_corrente.ux, particella_corrente.uy, particella_corrente.uz, cos_theta, phi);

                    if (particella_corrente.energia < ECUT) {
                        dose[id_voxel] += particella_corrente.energia;
                        break;
                    }
                }

                // PRODUZIONE DI COPPIE
                else {
                    // Energia cinetica disponibile per elettrone e positrone
                    double energia_cinetica_residua = particella_corrente.energia - 2.0 * ME_C2;
                    if (energia_cinetica_residua > 0.0) {
                        dose[id_voxel] += energia_cinetica_residua;
                    }

                    if (ME_C2 > ECUT && num_particelle_stack + 2 <= 62) {
                        double cos_theta = 2.0 * rng() - 1.0;
                        double phi_a = 2.0 * PI * rng();
                        double sen_theta = std::sqrt(std::max(0.0, 1.0 - cos_theta * cos_theta));

                        Particle fotone_secondario_1, fotone_secondario_2;
                        fotone_secondario_1.x = fotone_secondario_2.x = particella_corrente.x;
                        fotone_secondario_1.y = fotone_secondario_2.y = particella_corrente.y;
                        fotone_secondario_1.z = fotone_secondario_2.z = particella_corrente.z;
                        fotone_secondario_1.ux =  sen_theta * std::cos(phi_a);
                        fotone_secondario_1.uy =  sen_theta * std::sin(phi_a);
                        fotone_secondario_1.uz =  cos_theta;
                        fotone_secondario_2.ux = -fotone_secondario_1.ux;
                        fotone_secondario_2.uy = -fotone_secondario_1.uy;
                        fotone_secondario_2.uz = -fotone_secondario_1.uz;
                        fotone_secondario_1.energia = fotone_secondario_2.energia = ME_C2;

                        stack[num_particelle_stack++] = fotone_secondario_1;
                        stack[num_particelle_stack++] = fotone_secondario_2;
                    }
                    break;
                }

            } else {
                double eps = 1.0e-7;
                particella_corrente.x += particella_corrente.ux * (distanza_fisica + eps);
                particella_corrente.y += particella_corrente.uy * (distanza_fisica + eps);
                particella_corrente.z += particella_corrente.uz * (distanza_fisica + eps);
            }

        }
    }
}

int main(int argc, char *argv[]) {

    // valori default
    long long num_fotoni = 1000000;   // default: 1M fotoni
    int tipo_phantom = 0;         // 0=acqua, 1=eterogeneo
    uint64_t seed   = 42ULL;

    if (argc > 1) num_fotoni = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label;
    if (tipo_phantom == 0){
      phantom_label = "Acqua omogenea";
    } else{
      phantom_label = "Acqua + Osso";
    }

    printf("  Monte Carlo per Radioterapia — CPU Sequenziale\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f³ cm³\n", NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    int *phantom = new int [NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();
    double *pdd = new double[NZ];
    double *coordinate_cm = new double[NZ];
    double *profilo_dose = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    if (tipo_phantom == 0){
        printf("Costruzione phantom con acqua \n");
        build_phantom_water(phantom);
    }else {
        printf("Costruzione phantom eterogeneo \n");
        build_phantom_hetero(phantom);
    }

    Spectrum spettro;  // spettro 6MV con CDF
    Xoshiro256 rng(seed); // PRNG xoshiro256**

    printf(" Avvio simulazione \n");
    auto tempo_inizio_esatto = std::chrono::high_resolution_clock::now();

    long long report_step = std::max(1LL, num_fotoni / 20);   // report ogni 5%

    for (long long i = 0; i < num_fotoni; i++) {
        if ((i + 1) % report_step == 0) {
            auto tempo_attuale     = std::chrono::high_resolution_clock::now();
            double tempo_esecuzione = std::chrono::duration<double>(tempo_attuale - tempo_inizio_esatto).count();
            double rate  = (i + 1) / tempo_esecuzione;
            double tempo_necessrio_fotoni_rimanenti   = (num_fotoni - i - 1) / rate;
            printf(" [%5.1f%%]  %.0f fotoni/s  Estimated Time of Arrival %.0fs\n", 100.0 * (i + 1) / num_fotoni, rate, tempo_necessrio_fotoni_rimanenti);
        }

        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose, rng);
    }

    auto tempo_fine_esatto = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(tempo_fine_esatto - tempo_inizio_esatto).count();

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);

    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file;
    const char *profilo_file;
    const char *slice_file;
    const char *bin_file;

    if (tipo_phantom == 0){
      pdd_file = "pdd_water.csv";
      profilo_file = "profile_water.csv";
      slice_file = "dose_slice_water.csv";
      bin_file = "dose_water.bin";
    } else{
      pdd_file = "pdd_hetero.csv";
      profilo_file = "profile_hetero.csv";
      slice_file = "dose_slice_hetero.csv";
      bin_file = "dose_hetero.bin";
    }
    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom;
    delete[] dose;
    delete[] pdd;
    delete[] coordinate_cm;
    delete[] profilo_dose;
    delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");

    return 0;
}


Writing main.cpp


### File BeerLambert.cpp

E' stata implementata una versione semplificata di un simulatore Monte Carlo per radioterapia per validare la Legge di Beer-Lambert. Viene simulato il trasporto di fotoni (con uno spettro energetico da 6MV) attraverso un mezzo materiale per dimostrare che l'attenuazione della radiazione segua un decadimento esponenziale perfetto quando viene considerato solo l'assorbimento primario.

Per fare questo non appena il fotone interagisce con un voxel del phantom tutta la sua energia viene depositata immediatamente. Vengono ignorati i processi di scattering, come l'effetto Compton, dove il fotone cambierebbe solo direzione ed energia continuando il suo viaggio. In questo modo, il numero di fotoni che raggiungono una certa profondità dipende esclusivamente dalla probabilità di non aver interagito prima.

In [109]:
%%writefile BeerLambert.cpp

#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <chrono>
#include <cstring>
#include <vector>

#include "physics.h"
#include "compton.h"
#include "random.h"
#include "phantom.h"
#include "output.h"

// stato di una particella sullo stack
struct Particle {
    double x, y, z;    // posizione [cm]
    double ux, uy, uz; // versore direzione (normalizzato)
    double energia;     // energia [MeV]
};

// CAMPIONAMENTO SORGENTE
inline Particle sample_source(const Spectrum &spettro, Xoshiro256 &rng) {
    static const double FIELD_HALF = 5.0;   // +-5 cm -> campo 10×10 cm^2
    double cx = PHANTOM_CM / 2.0;
    double cy = PHANTOM_CM / 2.0;

    Particle p;
    p.x = cx + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.y = cy + (rng() * 2.0 - 1.0) * FIELD_HALF;
    p.z = 1.0e-7;
    p.ux = 0.0;
    p.uy = 0.0;
    p.uz = 1.0;
    p.energia = spettro.sample(rng);
    return p;
}

// DISTANZA AL PROSSIMO CONFINE DI VOXEL
inline double boundary_distance(double x, double y, double z, double ux, double uy, double uz, int ix, int iy, int iz) {
    double distanza_minima_confine = 1.0e30; // inizializzata a infinito
    if (std::fabs(ux) > 1.0e-12) {
        double confine_voxel_X;
        if (ux > 0){
            confine_voxel_X = (ix + 1) * VOXEL_CM;
        } else{
            confine_voxel_X = ix * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_X - x) / ux;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uy) > 1.0e-12) {
        double confine_voxel_Y;
        if (uy > 0){
            confine_voxel_Y = (iy + 1) * VOXEL_CM;
        } else{
            confine_voxel_Y = iy * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Y - y) / uy;
        if (distanza_lineare > 1.0e-10){
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    if (std::fabs(uz) > 1.0e-12) {
        double confine_voxel_Z;
        if (uz > 0){
            confine_voxel_Z = (iz + 1) * VOXEL_CM;
        } else{
            confine_voxel_Z = iz * VOXEL_CM;
        }
        double distanza_lineare = (confine_voxel_Z - z) / uz;
        if (distanza_lineare > 1.0e-10) {
            distanza_minima_confine = std::min(distanza_minima_confine, distanza_lineare);
        }
    }
    return distanza_minima_confine;
}

// TRASPORTO FOTONE SEMPLIFICATO PER BEER LAMBERT
void transport_photon(Particle p, const int *phantom, double *dose, Xoshiro256 &rng) {
    while (p.energia > ECUT && inside(p.x, p.y, p.z)) {
        int mat = phantom[phantom_idx((int)(p.x/VOXEL_CM), (int)(p.y/VOXEL_CM), (int)(p.z/VOXEL_CM))];
        double mu = get_mu_total(p.energia, mat);
        double d = -std::log(rng()) / mu;

        p.x += p.ux * d;
        p.y += p.uy * d;
        p.z += p.uz * d;

        if (inside(p.x, p.y, p.z)) {
            int ix = (int)(p.x / VOXEL_CM);
            int iy = (int)(p.y / VOXEL_CM);
            int iz = (int)(p.z / VOXEL_CM);

            dose[phantom_idx(ix, iy, iz)] += p.energia;

            break;
        }
    }
}

int main(int argc, char *argv[]) {

    // valori default
    long long num_fotoni = 1000000;   // default: 1M fotoni
    int tipo_phantom = 0;         // 0=acqua, 1=eterogeneo
    uint64_t seed   = 42ULL;

    if (argc > 1) num_fotoni = std::atoll(argv[1]);
    if (argc > 2) tipo_phantom = std::atoi(argv[2]);
    if (argc > 3) seed = (uint64_t)std::atoll(argv[3]);

    const char *phantom_label;
    if (tipo_phantom == 0){
      phantom_label = "Acqua omogenea";
    } else{
      phantom_label = "Acqua + Osso";
    }

    printf("  Monte Carlo per Radioterapia — CPU Sequenziale\n");
    printf("\n  Parametri:\n");
    printf("  Phantom    : %dx%dx%d voxel  |  voxel %.0fmm  |  %.0f³ cm³\n", NX, NY, NZ, VOXEL_CM * 10.0, PHANTOM_CM);
    printf("  Materiale  : %s\n", phantom_label);
    printf("  N fotoni   : %lld\n", num_fotoni);
    printf("  Seed       : %llu\n", (unsigned long long)seed);
    printf("  ECUT       : %.0f keV\n\n", ECUT * 1000.0);

    int *phantom = new int [NX * NY * NZ];
    double *dose = new double[NX * NY * NZ]();
    double *pdd = new double[NZ];
    double *coordinate_cm = new double[NZ];
    double *profilo_dose = new double[NX];
    double *coordinate_cm_laterali = new double[NX];

    if (tipo_phantom == 0){
        printf("Costruzione phantom con acqua \n");
        build_phantom_water(phantom);
    }else {
        printf("Costruzione phantom eterogeneo \n");
        build_phantom_hetero(phantom);
    }

    Spectrum spettro;  // spettro 6MV con CDF
    Xoshiro256 rng(seed); // PRNG xoshiro256**

    printf(" Avvio simulazione \n");
    auto tempo_inizio_esatto = std::chrono::high_resolution_clock::now();

    long long report_step = std::max(1LL, num_fotoni / 20);   // report ogni 5%

    for (long long i = 0; i < num_fotoni; i++) {
        if ((i + 1) % report_step == 0) {
            auto tempo_attuale     = std::chrono::high_resolution_clock::now();
            double tempo_esecuzione = std::chrono::duration<double>(tempo_attuale - tempo_inizio_esatto).count();
            double rate  = (i + 1) / tempo_esecuzione;
            double tempo_necessrio_fotoni_rimanenti   = (num_fotoni - i - 1) / rate;
            printf(" [%5.1f%%]  %.0f fotoni/s  Estimated Time of Arrival %.0fs\n", 100.0 * (i + 1) / num_fotoni, rate, tempo_necessrio_fotoni_rimanenti);
        }

        Particle p = sample_source(spettro, rng);
        transport_photon(p, phantom, dose, rng);
    }

    auto tempo_fine_esatto = std::chrono::high_resolution_clock::now();
    double tempo_esecuzione = std::chrono::duration<double>(tempo_fine_esatto - tempo_inizio_esatto).count();

    print_dose_stats(dose, num_fotoni, tempo_esecuzione);

    compute_pdd(dose, pdd, coordinate_cm);
    compute_lateral_profile(dose, profilo_dose, coordinate_cm_laterali, 10.0);
    print_pdd_table(coordinate_cm, pdd, phantom_label);

    const char *pdd_file;
    const char *profilo_file;
    const char *slice_file;
    const char *bin_file;

    if (tipo_phantom == 0){
      pdd_file = "pdd_water_BL.csv";
      profilo_file = "profile_water_BL.csv";
      slice_file = "dose_slice_water_BL.csv";
      bin_file = "dose_water_BL.bin";
    } else{
      pdd_file = "pdd_hetero_BL.csv";
      profilo_file = "profile_hetero_BL.csv";
      slice_file = "dose_slice_hetero_BL.csv";
      bin_file = "dose_hetero_BL.bin";
    }
    save_pdd_csv(coordinate_cm, pdd, pdd_file);
    save_profile_csv(coordinate_cm_laterali, profilo_dose, profilo_file);
    save_dose_slice_csv(dose, slice_file);
    save_dose_binary(dose, bin_file);

    delete[] phantom;
    delete[] dose;
    delete[] pdd;
    delete[] coordinate_cm;
    delete[] profilo_dose;
    delete[] coordinate_cm_laterali;

    printf("  Simulazione completata.\n");

    return 0;
}


Overwriting BeerLambert.cpp


## Compilazione

Compilazione main completo

In [9]:
!g++ -O2 -std=c++17 -o mc_rt_cpu main.cpp -lm

Compilazione main per validazione Beer Lambert

In [110]:
!g++ -O2 -std=c++17 -o test_beer_lambert BeerLambert.cpp -lm

## Esecuzione

In [70]:
numero_fotoni = 10000000

In [72]:
!./mc_rt_cpu $numero_fotoni 0 42

  Monte Carlo per Radioterapia — CPU Sequenziale

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30³ cm³
  Materiale  : Acqua omogenea
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

Costruzione phantom con acqua 
 Avvio simulazione 
 [  5.0%]  63799 fotoni/s  Estimated Time of Arrival 149s
 [ 10.0%]  61995 fotoni/s  Estimated Time of Arrival 145s
 [ 15.0%]  64163 fotoni/s  Estimated Time of Arrival 132s
 [ 20.0%]  57836 fotoni/s  Estimated Time of Arrival 138s
 [ 25.0%]  57274 fotoni/s  Estimated Time of Arrival 131s
 [ 30.0%]  56475 fotoni/s  Estimated Time of Arrival 124s
 [ 35.0%]  51878 fotoni/s  Estimated Time of Arrival 125s
 [ 40.0%]  51922 fotoni/s  Estimated Time of Arrival 116s
 [ 45.0%]  50191 fotoni/s  Estimated Time of Arrival 110s
 [ 50.0%]  50651 fotoni/s  Estimated Time of Arrival 99s
 [ 55.0%]  50069 fotoni/s  Estimated Time of Arrival 90s
 [ 60.0%]  48999 fotoni/s  Estimated Time of Arrival 82s
 [ 65.0%]  48590 fotoni/s  Estimated Ti

In [92]:
!./mc_rt_cpu $numero_fotoni 1 42

  Monte Carlo per Radioterapia — CPU Sequenziale

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30³ cm³
  Materiale  : Acqua + Osso
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

Costruzione phantom eterogeneo 
Inserto osseo: 4096 voxel = 110.6 cm³  ( volume teorico 125 cm³)
 Avvio simulazione 
 [  5.0%]  49039 fotoni/s  Estimated Time of Arrival 194s
 [ 10.0%]  52549 fotoni/s  Estimated Time of Arrival 171s
 [ 15.0%]  50013 fotoni/s  Estimated Time of Arrival 170s
 [ 20.0%]  46115 fotoni/s  Estimated Time of Arrival 173s
 [ 25.0%]  48928 fotoni/s  Estimated Time of Arrival 153s
 [ 30.0%]  47795 fotoni/s  Estimated Time of Arrival 146s
 [ 35.0%]  48043 fotoni/s  Estimated Time of Arrival 135s
 [ 40.0%]  47794 fotoni/s  Estimated Time of Arrival 126s
 [ 45.0%]  46267 fotoni/s  Estimated Time of Arrival 119s
 [ 50.0%]  45010 fotoni/s  Estimated Time of Arrival 111s
 [ 55.0%]  44085 fotoni/s  Estimated Time of Arrival 102s
 [ 60.0%]  43698 fotoni/s  Est

In [111]:
!./test_beer_lambert $numero_fotoni 0 42

  Monte Carlo per Radioterapia — CPU Sequenziale

  Parametri:
  Phantom    : 100x100x100 voxel  |  voxel 3mm  |  30³ cm³
  Materiale  : Acqua omogenea
  N fotoni   : 10000000
  Seed       : 42
  ECUT       : 10 keV

Costruzione phantom con acqua 
 Avvio simulazione 
 [  5.0%]  7707769 fotoni/s  Estimated Time of Arrival 1s
 [ 10.0%]  7307102 fotoni/s  Estimated Time of Arrival 1s
 [ 15.0%]  7345318 fotoni/s  Estimated Time of Arrival 1s
 [ 20.0%]  7222298 fotoni/s  Estimated Time of Arrival 1s
 [ 25.0%]  7075674 fotoni/s  Estimated Time of Arrival 1s
 [ 30.0%]  7013448 fotoni/s  Estimated Time of Arrival 1s
 [ 35.0%]  6956019 fotoni/s  Estimated Time of Arrival 1s
 [ 40.0%]  6993619 fotoni/s  Estimated Time of Arrival 1s
 [ 45.0%]  6906310 fotoni/s  Estimated Time of Arrival 1s
 [ 50.0%]  6973899 fotoni/s  Estimated Time of Arrival 1s
 [ 55.0%]  6813829 fotoni/s  Estimated Time of Arrival 1s
 [ 60.0%]  6857181 fotoni/s  Estimated Time of Arrival 1s
 [ 65.0%]  6908864 fotoni/s  Estimat

# Test

## Costanti e metodi di utilità

In [123]:
import os, subprocess, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from pathlib import Path

NX = NY = NZ = 100
VOXEL_CM = 0.30
PHANTOM_CM = NX * VOXEL_CM
MASSA_ELETTRONE_MEV = 0.51099895
N = numero_fotoni

ESEGUIBILE_MC    = "./mc_rt_cpu"
DOSE_WATER    = "dose_water.bin"
DOSE_HETERO   = "dose_hetero.bin"
PDD_WATER     = "pdd_water.csv"
PDD_HETERO    = "pdd_hetero.csv"

PDD_WATER_BL     = "pdd_water_BL.csv"


# Definizione spettro 6MV (Sheikh-Bagheri e Rogers)
ENERGIE_MEV = np.array([0.25,0.50,0.75,1.00,1.25,1.50,1.75,2.00,
                   2.25,2.50,2.75,3.00,3.25,3.50,3.75,4.00,
                   4.25,4.50,4.75,5.00,5.25,5.50,5.75,6.00])
PESI_SPETTRO = np.array([0.0243,0.0676,0.0862,0.0929,0.0919,0.0868,0.0794,0.0712,
                   0.0628,0.0548,0.0471,0.0399,0.0334,0.0276,0.0224,0.0178,
                   0.0138,0.0104,0.0075,0.0052,0.0034,0.0020,0.0010,0.0004])
ENERGIA_MEDIA_6MV = float(np.sum(ENERGIE_MEV * PESI_SPETTRO) / np.sum(PESI_SPETTRO))

# Definizione riferimento PDD letteratura (Sheikh-Bagheri e Rogers)
PROFONDITA_RIF = np.array([0.15,0.30,0.45,0.60,0.75,0.90,1.05,1.20,1.35,1.50,
                    1.80,2.10,2.40,3.00,3.60,4.20,5.00,6.00,7.00,8.00,
                    9.00,10.0,11.0,12.0,13.0,14.0,15.0,16.0,17.0,18.0,
                    19.0,20.0,21.0,22.0,23.0,24.0,25.0])
DOSE_RIF = np.array([74.0,82.0,88.0,93.0,96.5,98.5,99.5,100.0,99.8,99.5,
                    98.5,97.5,96.5,94.5,92.5,90.5,87.5,83.5,79.5,76.0,
                    73.0,70.0,67.0,64.0,61.0,58.5,56.0,53.5,51.0,48.5,
                    46.5,44.5,42.5,40.5,38.5,37.0,35.5])

def lancia_simulazione(n_fotoni, tipo_fantoccio=0, seed=42):
    comando = [ESEGUIBILE_MC, str(int(n_fotoni)), str(tipo_fantoccio), str(seed)]
    inizio = time.time()
    risultato = subprocess.run(comando, capture_output=True, text=True)
    durata = time.time() - inizio
    if risultato.returncode != 0:
        print(f"Errore nella simulazione: {risultato.stderr[:200]}")
        return False
    return True


def load_pdd(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None, None
    data = np.loadtxt(filename, delimiter=',', skiprows=1)
    return data[:, 0], data[:, 1]

def load_dose_bin(filename):
    if not os.path.exists(filename):
        print(f"  [ERRORE] file non trovato: {filename}"); return None
    flat = np.fromfile(filename, dtype=np.float64)
    if flat.size != NX * NY * NZ:
        print(f"  [ERRORE] dimensione inattesa: {flat.size}"); return None
    return flat.reshape((NZ, NY, NX))

def save_fig(fig, name):
    path = f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f"    Figura salvata: {path}")

def kn_analytic_mean_cos(energy_mev, n_pts=20_000):
    alpha = energy_mev / MASSA_ELETTRONE_MEV
    ct    = np.linspace(-1.0, 1.0, n_pts)
    tau   = 1.0 / (1.0 + alpha * (1.0 - ct))
    kn    = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + ct**2)
    kn   /= np.trapz(kn, ct)
    return float(np.trapz(ct * kn, ct))

def kahn_sample_python(energy_mev, rng, n=50_000):
    alpha, tau_min = energy_mev / MASSA_ELETTRONE_MEV, 1.0 / (1.0 + 2.0 * energy_mev / MASSA_ELETTRONE_MEV)
    a1  = np.log(1.0 / tau_min)
    a2  = (1.0 - tau_min**2) * 0.5
    a12 = a1 + a2
    cos_thetas, total = [], 0
    while len(cos_thetas) < n:
        total += 1
        xi1, xi2, xi3 = rng.random(3)
        tau = tau_min**(1.0-xi2) if xi1*a12 < a1 else np.sqrt(tau_min**2 + xi2*(1.0-tau_min**2))
        tau = float(np.clip(tau, tau_min, 1.0))
        ct  = float(np.clip(1.0 - (1.0 - tau) / (alpha * tau), -1.0, 1.0))
        sin2 = max(0.0, 1.0 - ct**2)
        g = max(0.0, min(1.0, 1.0 - tau * sin2 / (1.0 + tau**2)))
        if xi3 <= g:
            cos_thetas.append(ct)
    return np.array(cos_thetas), len(cos_thetas)/total*100.0

## Test 1 - Analisi del Coefficiente di Attenuazione Lineare ($\mu$)

Viene eseguita la validazione fisica della distribuzione di dose in acqua attraverso il calcolo del coefficiente di attenuazione lineare. Questo serve per verificare che il comportamento dei fotoni simulati segua correttamente la legge dell'attenuazione esponenziale nella regione di post-massimo.

Viene selezionata la regione (ROI) compresa tra 5.0 cm e 20.0 cm perchè è quella dove l'attenuazione è prevalentemente dominata dalla componente primaria del fascio. Viene eseguita una regressione lineare sul logaritmo della dose rispetto alla profondità. La pendenza della retta risultante identifica il coefficiente $\mu$ misurato. Viene, inoltre, calcolato il coefficiente di determinazione per valutare la bontà del fit e la coerenza del modello esponenziale con i dati simulati. Il valore ottenuto viene confrontato con un range di accettabilità fisica e confrontato con i dati standard NIST XCOM per energie monoenergetiche.

In [31]:
def test_1_coefficiente_attenuazione():

    profondita, dose = load_pdd(PDD_WATER)
    if profondita is None:
        return False

    filtro_zona_fit = (profondita >= 5.0) & (profondita <= 20.0) & (dose > 0)
    z_da_analizzare = profondita[filtro_zona_fit]
    dose_da_analizzare = dose[filtro_zona_fit]

    pendenza, intercetta = np.polyfit(z_da_analizzare, np.log(dose_da_analizzare), 1)
    mu_misurato = -pendenza

    MU_MIN = 0.030
    MU_MAX = 0.055

    test_superato = MU_MIN <= mu_misurato <= MU_MAX

    residui = np.log(dose_da_analizzare) - (pendenza * z_da_analizzare + intercetta)
    varianza_totale = np.log(dose_da_analizzare) - np.log(dose_da_analizzare).mean()

    somma_residui = np.sum(residui**2)
    somma_totale = np.sum(varianza_totale**2)
    r_quadrato = 1.0 - (somma_residui / somma_totale) if somma_totale > 0 else 0.0

    print(f"\n ANALISI DEL COEFFICIENTE DI ATTENUAZIONE ")
    print(f" Valore calcolato:  {mu_misurato:.5f} cm⁻¹")
    print(f" Qualità estrazione:   {r_quadrato:.4f}")


    if test_superato:
        print(f"\n Verifica limite fisico (Range atteso: {MU_MIN} - {MU_MAX} cm⁻¹) coerente")
    else:
        print(f" Valore è fuori range")

    print(f"\n Confronto con dati standard (NIST XCOM Monoenergetici):")
    dati_riferimento_nist = {
        "1.0 MeV": 0.07072,
        "1.5 MeV": 0.05754,
        "2.0 MeV": 0.04942,
        "3.0 MeV": 0.03969
    }

    for energia, mu_nist in dati_riferimento_nist.items():
        differenza_perc = abs(mu_misurato - mu_nist) / mu_nist * 100
        print(f" Rispetto a fotoni da {energia}:")
        print(f"  - Valore teorico: {mu_nist:.5f} cm⁻¹")
        print(f"  - Valore misurato: {mu_misurato:.5f} cm⁻¹")
        print(f"  - Differenza:     {differenza_perc:.1f}%")
    print("\n")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.semilogy(profondita, dose, 'b-', label='Dose Simulata')
    z_grafico = np.linspace(0, 30, 100)
    dose_fit = np.exp(intercetta + pendenza * z_grafico)
    ax.semilogy(z_grafico, dose_fit, 'r--', label=f'Fit Esponenziale (μ={mu_misurato:.4f})')
    ax.axvspan(5.0, 20.0, color='green', alpha=0.1, label='Zona Analisi')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title(f'Validazione Attenuazione - {"Superato" if test_superato else "Non superato"}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    save_fig(fig, 'Test_1_Valutazione_Attenuazione')

    print(f"\n TEST 1 SUPERATO: {test_superato}")

    return test_superato


risultato = test_1_coefficiente_attenuazione()


 ANALISI DEL COEFFICIENTE DI ATTENUAZIONE 
 Valore calcolato:  0.03961 cm⁻¹
 Qualità estrazione:   0.9978

 Verifica limite fisico (Range atteso: 0.03 - 0.055 cm⁻¹) coerente

 Confronto con dati standard (NIST XCOM Monoenergetici):
 Rispetto a fotoni da 1.0 MeV:
  - Valore teorico: 0.07072 cm⁻¹
  - Valore misurato: 0.03961 cm⁻¹
  - Differenza:     44.0%
 Rispetto a fotoni da 1.5 MeV:
  - Valore teorico: 0.05754 cm⁻¹
  - Valore misurato: 0.03961 cm⁻¹
  - Differenza:     31.2%
 Rispetto a fotoni da 2.0 MeV:
  - Valore teorico: 0.04942 cm⁻¹
  - Valore misurato: 0.03961 cm⁻¹
  - Differenza:     19.8%
 Rispetto a fotoni da 3.0 MeV:
  - Valore teorico: 0.03969 cm⁻¹
  - Valore misurato: 0.03961 cm⁻¹
  - Differenza:     0.2%


    Figura salvata: Test_1_Valutazione_Attenuazione.png

 TEST 1 SUPERATO: True


## Test 2 - Validazione della Legge di Beer Lambert

Viene eseguito un confronto tra i dati di dose ottenuti tramite simulazione Monte Carlo e il modello teorico basato sulla legge di Beer-Lambert. Il test verifica se il trasporto dei fotoni nel mezzo segue le leggi della fisica classica dell'attenuazione.

Viene generata una curva teorica ideale utilizzando il coefficiente di attenuazione standard ($\mu_{teorico} = 0.04942 \text{ cm}^{-1}$), corrispondente a fotoni monoenergetici da 2.0 MeV in acqua (dati NIST). Ogni punto viene considerato valido se l'errore assoluto rimane entro una tolleranza del 2.0%. Viene poi ricalcolato il coefficiente di attenuazione dalla simulazione nella zona di equilibrio e confrontato con il valore teorico. Il test viene superato solo se l'errore percentuale su $\mu$ è inferiore al 2%.

In [113]:
def test_2_validazione_BeerLambert():

    MU_TEORICO = 0.04942
    TOLLERANZA_DOSE = 2.0

    profondita, dose_mc = load_pdd(PDD_WATER_BL)
    if profondita is None:
        return False

    dose_teorica = 100.0 * np.exp(-MU_TEORICO * profondita)

    print(f"\n Confronto dose simulata e dose teorica:")
    print(f"{'Prof [cm]':>10} | {'Simulata [%]':>12} | {'Teorica [%]':>12} | {'Errore':>8}")
    print("-" * 55)

    esiti_punti = []
    profondita_controllo = [1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0]

    for z in profondita_controllo:
        idx = np.abs(profondita - z).argmin()
        d_sim = dose_mc[idx]
        d_teo = 100.0 * np.exp(-MU_TEORICO * z)
        differenza = abs(d_sim - d_teo)

        punto_ok = differenza < TOLLERANZA_DOSE
        esiti_punti.append(punto_ok)

        stato = "BUONO" if punto_ok else "ERRORE"
        print(f"{z:>10.1f} | {d_sim:>12.2f} | {d_teo:>12.2f} | {stato:>8} ({differenza:.3f})")

    filtro = (profondita >= 5.0) & (profondita <= 20.0)
    pendenza, _ = np.polyfit(profondita[filtro], np.log(dose_mc[filtro]), 1)
    mu_misurato = -pendenza
    errore_mu = abs(mu_misurato - MU_TEORICO) / MU_TEORICO * 100

    mu_ok = errore_mu < 2.0
    esiti_punti.append(mu_ok)

    print(f"\n Analisi Coefficiente (mu):")
    print(f"  Misurato: {mu_misurato:.5f} cm⁻¹")
    print(f"  Teorico:  {MU_TEORICO:.5f} cm⁻¹")
    print(f"  Errore:   {errore_mu:.2f}% ({'Nei limiti' if mu_ok else 'Fuori limite'})")
    print("\n")

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(profondita, dose_mc, 'b-', label='Simulazione Monte Carlo')
    ax.plot(profondita, dose_teorica, 'r--', label='Legge Teorica (Beer-Lambert)')
    ax.fill_between(profondita, dose_teorica - TOLLERANZA_DOSE, dose_teorica + TOLLERANZA_DOSE, color='red', alpha=0.1, label='Fascia Tolleranza ±2%')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Test 2: Validazione Fisica Beer Lambert')
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'Test_2_Validazione_Beer_Lambert')

    test_superato = all(esiti_punti)
    print(f"\nTEST 2 SUPERATO: {test_superato}")

    return test_superato

risultato = test_2_validazione_BeerLambert()



 Confronto dose simulata e dose teorica:
 Prof [cm] | Simulata [%] |  Teorica [%] |   Errore
-------------------------------------------------------
       1.5 |        93.02 |        92.86 |    BUONO (0.164)
       3.0 |        86.83 |        86.22 |    BUONO (0.605)
       5.0 |        76.93 |        78.11 |    BUONO (1.172)
      10.0 |        59.95 |        61.01 |    BUONO (1.059)
      15.0 |        47.47 |        47.65 |    BUONO (0.175)
      20.0 |        37.07 |        37.22 |    BUONO (0.145)
      25.0 |        29.15 |        29.07 |    BUONO (0.077)

 Analisi Coefficiente (mu):
  Misurato: 0.04938 cm⁻¹
  Teorico:  0.04942 cm⁻¹
  Errore:   0.09% (Nei limiti)


    Figura salvata: Test_2_Validazione_Beer_Lambert.png

TEST 2 SUPERATO: True


## Test 3 - Verifica delle sezioni d'urto e probabilità di interazione

Viene effettuata un'analisi  dei meccanismi di interazione radiazione-materia per fotoni da 2 MeV in acqua, sulla base dei dati del database NIST XCOM. Questo viene fatto per confermare che il trasporto dei fotoni nel simulatore rispetti la predominanza dei processi fisici attesi a questa specifica energia.

Per fare questo il coefficiente di attenuazione lineare totale ($\mu$) viene scomposto nei suoi tre contributi principali:
* Effetto Fotoelettrico
* Effetto Compton
* Produzione di Coppie

A 2 MeV in acqua, l'effetto Compton deve rappresentare la quasi totalità delle interazioni. Viene verificato che l'effetto fotoelettrico sia trascurabile e che la produzione di coppie sia coerente con l'energia del fascio impostata. Viene anche effettuato un controllo sulla conservazione della probabilità affinché la somma dei contributi parziali sia esattamente pari al 100%.

In [31]:
def test_3_verifica_sezioni_urto_nist():

    mu_fotoelettrico = 2.200e-7
    mu_compton = 0.04942
    mu_coppie = 0.0

    mu_totale = mu_fotoelettrico + mu_compton + mu_coppie

    perc_fotoelettrico = (mu_fotoelettrico / mu_totale) * 100
    perc_compton = (mu_compton / mu_totale) * 100
    perc_coppie = (mu_coppie / mu_totale) * 100

    print(f"ANALISI PROBABILITÀ DI INTERAZIONE (2 MeV in Acqua)\n")
    print(f"Coefficiente totale (mu): {mu_totale:.6f} cm²/g\n")
    print(f"Effetto Fotoelettrico   : {perc_fotoelettrico:.5f}%  (Atteso: quasi 0%)")
    print(f"Effetto Compton         : {perc_compton:.4f}% (Atteso: > 99.9%)")
    print(f"Produzione di Coppie    : {perc_coppie:.4f}%  (Atteso: 0.0%)")

    controlli = [
        ("Dominanza Compton (> 99.9%)      ",  perc_compton > 99.9),
        ("Fotoelettrico trascurabile       ",   perc_fotoelettrico < 0.01),
        ("Assenza produzione coppie        ",    perc_coppie == 0.0),
        ("Coerenza matematica (Somma=100%) ", abs(perc_fotoelettrico + perc_compton + perc_coppie - 100.0) < 1e-6)
    ]

    esiti = []
    print("\nVerifica coerenza dati:")
    for descrizione, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {descrizione}: {stato}")
        esiti.append(superato)

    test_superato = all(esiti)
    print(f"\nTEST 3 SUPERATO: {test_superato}")

    return test_superato

risultato = test_3_verifica_sezioni_urto_nist()

ANALISI PROBABILITÀ DI INTERAZIONE (2 MeV in Acqua)

Coefficiente totale (mu): 0.049420 cm²/g

Effetto Fotoelettrico   : 0.00045%  (Atteso: quasi 0%)
Effetto Compton         : 99.9996% (Atteso: > 99.9%)
Produzione di Coppie    : 0.0000%  (Atteso: 0.0%)

Verifica coerenza dati:
  Dominanza Compton (> 99.9%)      : Coerente
  Fotoelettrico trascurabile       : Coerente
  Assenza produzione coppie        : Coerente
  Coerenza matematica (Somma=100%) : Coerente

TEST 3 SUPERATO: True


## Test 4 - Validazione stocastica della deviazione Compton


Viene verificata l'accuratezza dell'algoritmo di campionamento stocastico per lo scattering Compton. Nello specifico, confronta l'implementazione numerica basata sull'algoritmo di Kahn (metodo di rigetto) con la distribuzione teorica predetta dalla formula di Klein-Nishina.

Per diverse energie del fotone incidente (da 0.5 a 6.0 MeV), vengono generati 50.000 campioni dell'angolo di deviazione ($\cos\theta$) utilizzando il generatore di numeri casuali. Per ogni energia, poi, viene calcolata la media dei coseni campionati e confrontata con il valore medio analitico derivato dall'integrazione della sezione d'urto di Klein-Nishina. Il test è superato se l'errore relativo sulla media dei coseni rimane inferiore al 5% per tutti i livelli energetici testati.

In [42]:
def test_4_validazione_compton_kahn():

    energie_test = [0.5, 1.0, 2.0, 4.0, 6.0]
    generatore_casuale = np.random.default_rng(42)
    numero_campioni = 50_000
    limite_errore = 5.0

    print(f"ANALISI DEVIAZIONE COMPTON (Kahn e Klein-Nishina)\n")
    print(f"{'E [MeV]':>8} | {'<cos> MC':>12} | {'<cos> Teorico':>14} | {'Errore %':>10}  | {'Efficienza':>10}")
    print("-" * 67)

    esiti = []
    dati_per_grafico = {}

    for E in energie_test:
        media_teorica = kn_analytic_mean_cos(E)
        angoli_campionati, efficienza = kahn_sample_python(E, generatore_casuale, n=numero_campioni)

        media_mc = float(angoli_campionati.mean())
        errore_relativo = abs(media_mc - media_teorica) / abs(media_teorica) * 100.0

        ok = errore_relativo < limite_errore
        esiti.append(ok)
        dati_per_grafico[E] = angoli_campionati

        stato = "OK" if ok else "KO"
        print(f"{E:>8.1f} | {media_mc:>12.4f} | {media_teorica:>14.4f} | {errore_relativo:>9.2f}%  | {efficienza:.3f}")
    print("")

    E_plot = 2.0
    campioni_2mev = dati_per_grafico[E_plot]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(campioni_2mev, bins=50, density=True, alpha=0.5, color='steelblue', label=f'Simulazione Kahn ({numero_campioni} urti)')
    cos_grid = np.linspace(-1, 1, 400)

    alfa = E_plot / 0.511
    tau = 1.0 / (1.0 + alfa * (1.0 - cos_grid))
    pdf_teorica = 0.5 * tau**2 * (tau + 1.0/tau - 1.0 + cos_grid**2)
    pdf_teorica /= np.trapz(pdf_teorica, cos_grid)

    ax.plot(cos_grid, pdf_teorica, 'r-', lw=2, label='Teoria Klein-Nishina')
    ax.set_xlabel('Coseno dell\'angolo di deviazione (cos theta)')
    ax.set_ylabel('Probabilità (PDF)')
    ax.set_title(f'Confronto Distribuzione Angolare a {E_plot} MeV')
    ax.legend()
    ax.grid(True, alpha=0.3)

    save_fig(fig, 'Test_4_validazione_compton_kahn')

    test_superato = all(esiti)
    print(f"\nTEST 4 SUPERATO: {test_superato}")

    return test_superato

risultato = test_4_validazione_compton_kahn()

ANALISI DEVIAZIONE COMPTON (Kahn e Klein-Nishina)

 E [MeV] |     <cos> MC |  <cos> Teorico |   Errore %  | Efficienza
-------------------------------------------------------------------
     0.5 |       0.2863 |         0.2891 |      0.97%  | 73.958
     1.0 |       0.3613 |         0.3602 |      0.30%  | 79.916
     2.0 |       0.4232 |         0.4256 |      0.57%  | 86.032
     4.0 |       0.4838 |         0.4870 |      0.66%  | 90.843
     6.0 |       0.5222 |         0.5208 |      0.26%  | 93.223

    Figura salvata: Test_4_validazione_compton_kahn.png

TEST 4 SUPERATO: True


## Test 5 - Analisi del bilancio energetico globale

Viene verificata la coerenza termodinamica della simulazione, analizzando come l'energia emessa dalla sorgente (spettro clinico da 6 MV) venga distribuita all'interno del phantom d'acqua. L'obiettivo è confermare che non vi siano perdite ingiustificate o creazioni di energia non fisica durante il trasporto dei voxel.

Viene calcoalta l'energia totale immessa nel sistema che viene confrontata con la somma integrale della dose depositata in tutto il volume 3D. In un phantom di dimensioni finite (30 cm), è normale che una parte dell'energia venga trasportata all'esterno dai fotoni trasmessi o diffusi (backscattering e leakage laterale). La frazione depositata deve rientrare in un intervallo realistico per un fascio da 6 MV.

In [77]:
def test_5_bilancio_energetico():

    print(f"ANALISI DEL BILANCIO ENERGETICO (Spettro 6MV)")

    energia_media_singolo_fotone = ENERGIA_MEDIA_6MV
    dose_3d = load_dose_bin(DOSE_WATER)
    if dose_3d is None:
        return False

    energia_totale_depositata = float(dose_3d.sum())
    energia_totale_emessa = N * energia_media_singolo_fotone
    frazione_depositata = energia_totale_depositata / energia_totale_emessa

    voxel_colpiti = int((dose_3d > 0).sum())
    percentuale_volume = (voxel_colpiti / dose_3d.size) * 100

    print(f"\nRisultati energetici:")
    print(f"  Energia emessa dalla sorgente:  {energia_totale_emessa:.4e} MeV")
    print(f"  Energia rimasta nel fantoccio:  {energia_totale_depositata:.4e} MeV")
    print(f"  Frazione di energia catturata:  {frazione_depositata*100:.1f}%")
    print(f"  Volume d'acqua interessato   :     {percentuale_volume:.1f}% dei voxel")

    controlli = [
        ("Deposizione di energia :", energia_totale_depositata > 0),
        ("Conservazione energia  :", frazione_depositata < 1.0),
        ("Range fisico atteso    :", 0.30 < frazione_depositata < 0.90)
    ]

    esiti = []
    print("\nVerifica coerenza fisica:")
    for descrizione, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {descrizione} {stato}")
        esiti.append(superato)
    print()

    fig, (ax_profilo, ax_barre) = plt.subplots(1, 2, figsize=(12, 5))
    energia_per_fetta_z = dose_3d.sum(axis=(1, 2))
    profondita = np.linspace(0, 30, len(energia_per_fetta_z))
    ax_profilo.plot(profondita, energia_per_fetta_z, 'b-', lw=2)
    ax_profilo.set_xlabel('Profondità (cm)')
    ax_profilo.set_ylabel('Energia per fetta (MeV)')
    ax_profilo.set_title('Distribuzione energia lungo l\'asse')
    ax_profilo.grid(True, alpha=0.3)

    energia_persa = energia_totale_emessa - energia_totale_depositata
    ax_barre.bar(['Catturata', 'Uscita'], [energia_totale_depositata, energia_persa], color=['#4682B4', '#CD5C5C'], edgecolor='black', width=0.5)
    ax_barre.set_ylabel('Energia Totale (MeV)')
    ax_barre.set_title(f'Bilancio finale (Resa: {frazione_depositata*100:.1f}%)')
    ax_barre.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    save_fig(fig, 'Test_5_Validazione_bilancio_energia')

    test_superato = all(esiti)
    print(f"\nTEST 5 SUPERATO: {test_superato}")


    return test_superato
risultato = test_5_bilancio_energetico()

ANALISI DEL BILANCIO ENERGETICO (Spettro 6MV)

Risultati energetici:
  Energia emessa dalla sorgente:  1.9118e+07 MeV
  Energia rimasta nel fantoccio:  1.0509e+07 MeV
  Frazione di energia catturata:  55.0%
  Volume d'acqua interessato   :     99.9% dei voxel

Verifica coerenza fisica:
  Deposizione di energia : Coerente
  Conservazione energia  : Coerente
  Range fisico atteso    : Coerente

    Figura salvata: Test_5_Validazione_bilancio_energia.png

TEST 5 SUPERATO: True


## Test 6 - Validazione della forma del PDD

Viene valutata la qualità dosimetrica della simulazione confrontando il Percent Depth Dose (PDD) ottenuto con i parametri clinici standard e i dati di riferimento presenti in letteratura (Sheikh-Bagheri). Questo perchè si vuole garantire che il fascio simulato sia rappresentativo di un acceleratore lineare clinico reale. Viene individuata la profondità del massimo di dose che per un fascio da 6 MV deve avere un picco che si trovi tra 0 e 2 cm. La curva simulata viene confrontata punto per punto con dati di riferimento tramite un'interpolazione cubica.

In [88]:
def test_6_forma_pdd_clinico():

    print(f"ANALISI CLINICA PDD (Spettro 6MV)")

    profondita, dose = load_pdd(PDD_WATER)
    if profondita is None:
        return False

    indice_picco = np.argmax(dose)
    z_picco = profondita[indice_picco]
    valore_picco = dose[indice_picco]

    dose_10 = dose[np.abs(profondita - 10.0).argmin()]
    dose_20 = dose[np.abs(profondita - 20.0).argmin()]

    rapporto_10_max = dose_10 / valore_picco
    rapporto_20_10  = dose_20 / dose_10

    controlli = [
        ("Profondità del picco (0-2 cm) :", 0.0 <= z_picco <= 2.0, f"{z_picco:.2f} cm"),
        ("Rapporto Dose 10cm / Max      :", 0.60 <= rapporto_10_max <= 0.82, f"{rapporto_10_max:.3f}"),
        ("Rapporto Dose 20cm / 10cm     :", 0.45 <= rapporto_20_10 <= 0.72, f"{rapporto_20_10:.3f}")
    ]

    esiti = []
    print("\nVerifica parametri PDD:")
    for desc, superato, valore in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {desc} {valore} -> {stato}")
        esiti.append(superato)


    f_riferimento = interp1d(PROFONDITA_RIF, REF_PDD, kind='cubic', fill_value='extrapolate')

    zona = (profondita >= 1.5) & (profondita <= 25.0)
    differenze = np.abs(dose[zona] - f_riferimento(profondita[zona]))
    errore_medio = differenze.mean()

    giudizio = "Corretto" if errore_medio < 3.0 else "Accettabile" if errore_medio < 6.0 else "Errato"
    print(f"\nConfronto con dati storici (Sheikh-Bagheri):")
    print(f"  Errore medio (MAE): {errore_medio:.2f}% -> {giudizio}")
    print()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(profondita, dose, 'b-', lw=2.5, label='La tua Simulazione')
    ax.plot(PROFONDITA_RIF, REF_PDD, 'go', ms=5, label='Riferimento Letteratura')
    ax.axvline(z_picco, color='gray', ls='--', alpha=0.5, label=f'Picco a {z_picco:.1f} cm')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Confronto PDD: Simulazione e dati clinici')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 110)

    save_fig(fig, 'Test_6_confronto_pdd_clinico')

    test_superato = all(esiti)
    print(f"\nTEST 6 SUPERATO: {test_superato}")

    return test_superato

risultato = test_6_forma_pdd_clinico()

ANALISI CLINICA PDD (Spettro 6MV)

Verifica parametri PDD:
  Profondità del picco (0-2 cm) : 0.75 cm -> Coerente
  Rapporto Dose 10cm / Max      : 0.735 -> Coerente
  Rapporto Dose 20cm / 10cm     : 0.680 -> Coerente

Confronto con dati storici (Sheikh-Bagheri):
  Errore medio (MAE): 4.50% -> Accettabile

    Figura salvata: Test_7_confronto_pdd_clinico.png

TEST 7 SUPERATO: True


## Test 7 - Verifica dell'eterogeneità

Viene valutata la capacità di gestire variazioni di densità elettronica e numero atomico all'interno del volume. Viene analizzato il comportamento del fascio quando attraversa un inserto di osso compatto immerso in un phantom d'acqua.

Viene confrontato il profilo di dose in un fantoccio omogeneo (acqua) con uno eterogeneo contenente un inserto osseo posizionato tra 12.5 cm e 17.5 cm. All'interno dell'osso deve esserci una variazione della dose depositata rispetto all'acqua a causa del maggiore coefficiente di attenuazione e del diverso potere d'arresto del mezzo più denso. L'osso, attenuando il fascio più efficacemente dell'acqua, deve generare una riduzione significativa della dose a valle (un "effetto ombra" atteso > 3% a 20 cm di profondità).

In [93]:
def test_7_verifica_eterogeneita_osso():
    print(f"ANALISI ETEROGENEITÀ (Acqua-Osso)")

    prof_acqua, pdd_acqua = load_pdd(PDD_WATER)
    prof_osso, pdd_osso = load_pdd(PDD_HETERO)

    if prof_acqua is None or prof_osso is None:
      return False

    fetta_inizio, fetta_fine = int(12.5 / VOXEL_CM), int(17.5 / VOXEL_CM)
    dose_media_acqua = pdd_acqua[fetta_inizio:fetta_fine].mean()
    dose_media_osso  = pdd_osso[fetta_inizio:fetta_fine].mean()

    fetta_20cm = int(20.0 / VOXEL_CM)
    dose_dopo_acqua = pdd_acqua[fetta_20cm]
    dose_dopo_osso  = pdd_osso[fetta_20cm]

    diff_dentro = dose_media_osso - dose_media_acqua
    diff_dopo   = dose_dopo_acqua - dose_dopo_osso

    print(f"\nRisultati zona ossea (12.5-17.5 cm):")
    print(f"  Dose in acqua    : {dose_media_acqua:.2f}% | Dose in osso: {dose_media_osso:.2f}%")
    print(f"  Variazione locale: {diff_dentro:+.2f}% (Attesa: Positiva)")

    print(f"\nRisultati a valle (20 cm):")
    print(f"  Dose senza osso: {dose_dopo_acqua:.2f}% | Dose con osso: {dose_dopo_osso:.2f}%")
    print(f"  Effetto ombra  : {diff_dopo:+.2f}% (Atteso: > 3%)")

    controlli = [
        ("Maggiore assorbimento nell'osso", diff_dentro > 0),
        ("Presenza effetto ombra dopo l'osso", diff_dopo > 0),
        ("Entità dell'ombra significativa (>3%)", diff_dopo > 3.0)
    ]

    esiti = []
    print("\nVerifica fisica:")
    for desc, superato in controlli:
        stato = "Coerente" if superato else "Non coerente"
        print(f"  {desc} {stato}")
        esiti.append(superato)
    print()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(prof_acqua, pdd_acqua, 'b-', label='Tutta acqua')
    ax.plot(prof_osso, pdd_osso, 'r--', label='Con inserto osseo')
    ax.axvspan(12.5, 17.5, color='orange', alpha=0.2, label='Posizione Osso')
    ax.set_xlabel('Profondità (cm)')
    ax.set_ylabel('Dose (%)')
    ax.set_title('Test Eterogeneità: Effetto dell\'Osso sulla Dose')
    ax.legend()
    ax.grid(alpha=0.3)

    save_fig(fig, 'Test_7_Validazione_eterogeneita')

    test_superato = all(esiti)
    print(f"\nTEST 7 SUPERATO: {test_superato}")

    return test_superato

risultato = test_7_verifica_eterogeneita_osso()

ANALISI ETEROGENEITÀ (Acqua-Osso)

Risultati zona ossea (12.5-17.5 cm):
  Dose in acqua    : 62.02% | Dose in osso: 84.85%
  Variazione locale: +22.83% (Attesa: Positiva)

Risultati a valle (20 cm):
  Dose senza osso: 49.97% | Dose con osso: 34.04%
  Effetto ombra  : +15.92% (Atteso: > 3%)

Verifica fisica:
  Maggiore assorbimento nell'osso Coerente
  Presenza effetto ombra dopo l'osso Coerente
  Entità dell'ombra significativa (>3%) Coerente

    Figura salvata: Test_9_Validazione_eterogeneita.png

TEST 9 SUPERATO: True


## Test 8 - Analisi della penombra laterale e diffusione compton

Viene analizzata la qualità del fascio lungo l'asse trasversale con concentrazione principale sul fenomeno della penombra. L'obiettivo è quantificare come la nitidezza dei bordi del campo di radiazione degradi all'aumentare della profondità a causa dello scattering dei fotoni (principalmente diffusione Compton) e del trasporto degli elettroni secondari.

In [91]:
def test_8_verifica_penombra_laterale():

    print(f"ANALISI DIFFUSIONE LATERALE (Penombra Compton)")

    dose_3d = load_dose_bin(DOSE_WATER)
    if dose_3d is None:
      return False

    profondita_controllo = [3.0, 5.0, 10.0, 15.0, 20.0]
    misure_penombra = []
    profili_grafico = {}

    posizioni_x = (np.arange(NX) + 0.5) * VOXEL_CM - (PHANTOM_CM / 2.0)

    print(f"\n{'Profondità [cm]':>15} | {'Penombra [cm]':>15} | {'Andamento'}")
    print("-" * 50)

    for z in profondita_controllo:
        fetta_z = min(int(z / VOXEL_CM), NZ - 1)
        profilo = dose_3d[fetta_z, NY//2-4 : NY//2+5, :].mean(axis=0)
        profilo_pulito = np.convolve(profilo, np.ones(3)/3, mode='same')
        profilo_norm = (profilo_pulito / profilo_pulito.max()) * 100

        idx_80 = np.where(profilo_norm >= 80)[0][-1]
        idx_20 = np.where(profilo_norm >= 20)[0][-1]

        distanza = posizioni_x[idx_20] - posizioni_x[idx_80]
        misure_penombra.append(distanza)
        profili_grafico[z] = profilo_norm

        trend = ""
        if len(misure_penombra) < 2:
            trend = "In crescita"
        elif distanza >= misure_penombra[-2] - 0.05:
            trend = "Normale"
        else:
            trend = "Anomalo"

        print(f"{z:>15.1f} | {distanza:>15.2f} | {trend}")

    rapporto_20_3 = misure_penombra[-1] / misure_penombra[0]
    test_superato = rapporto_20_3 > 1.10

    print(f"\Risultati:")
    print(f"  Penombra a 3cm :  {misure_penombra[0]:.2f} cm")
    print(f"  Penombra a 20cm: {misure_penombra[-1]:.2f} cm")
    print(f"  Rapporto (20/3): {rapporto_20_3:.2f} (Target > 1.10)")
    print()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for z in profondita_controllo:
        ax1.plot(posizioni_x, profili_grafico[z], label=f"z = {z} cm")
    ax1.set_title("Sfumatura del bordo (Penombra)")
    ax1.set_xlabel("Posizione laterale (cm)")
    ax1.set_ylabel("Dose %")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(profondita_controllo, misure_penombra, 'ro-', lw=2)
    ax2.set_title("Allargamento del fascio")
    ax2.set_xlabel("Profondità (cm)")
    ax2.set_ylabel("Larghezza penombra (cm)")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    save_fig(fig, 'Test_8_Validazione_penombra_laterale')

    print(f"\nTEST 8 SUPERATO: {test_superato}")

    return test_superato
risultato = test_8_verifica_penombra_laterale()

ANALISI DIFFUSIONE LATERALE (Penombra Compton)

Profondità [cm] |   Penombra [cm] | Andamento
--------------------------------------------------
            3.0 |            0.60 | In crescita
            5.0 |            0.60 | Normale
           10.0 |            0.90 | Normale
           15.0 |            0.60 | Anomalo
           20.0 |            0.90 | Normale

Sintesi Risultati:
  Penombra a 3cm :  0.60 cm
  Penombra a 20cm: 0.90 cm
  Rapporto (20/3): 1.50 (Target > 1.10)

    Figura salvata: Test_8_Validazione_penombra_laterale.png

TEST 8 SUPERATO: True


## Test 9 - Analisi Gamma Index (2%/3mm)

Il Gamma Index è lo standard internazionale per il confronto quantitativo tra due distribuzioni di dose. Vengono integrate simultaneamente le differenze nella dose e le discrepanze spaziali fornendo un punteggio univoco che certifica l'accuratezza della simulazione rispetto allo standard di riferimento.

In [124]:
def test10_gamma_index(ref_csv: str | None = None) -> bool:
    """
    Confronto del PDD simulato con il riferimento di letteratura primaria:
      Sheikh-Bagheri D., Rogers D.W.O., Med. Phys. 29(3), 2002
      Fascio 6MV Varian 2100C, campo 10x10 cm2, SAD=100 cm, acqua.

    APPROCCIO: Strada 1 — confronto onesto con dichiarazione esplicita dei limiti.

    Il simulatore ha due limitazioni fisiche note rispetto a EGSnrc full-transport,
    entrambe documentate in letteratura e non imputabili a errori numerici:

    LIMITAZIONE 1 — KERMA electron approximation (zona build-up, z < 3 cm):
      Il simulatore deposita l'energia degli elettroni secondari nel punto di
      interazione fotonica (KERMA locale). EGSnrc trasporta gli elettroni per
      diversi mm, spostando il d_max da ~0 cm (KERMA) a ~1.2 cm (EGSnrc).
      Differenza sistematica nella zona di build-up: 5-26%.
      Rif.: Andreo P., Acta Oncol. 30(4), 1991; IAEA TRS-430, 2004.

    LIMITAZIONE 2 — Fascio parallelo vs divergente SAD=100 cm (z > 5 cm):
      La sorgente campiona fotoni con direzione uz=1 (fascio parallelo).
      Sheikh-Bagheri usa un fascio con divergenza reale SAD=100 cm, che riduce
      la fluenza alle grandi profondita per la legge inversa del quadrato.
      Con normalizzazione a D(max): sovrastima MC di ~5-6% a z > 10 cm.
      Con normalizzazione a D(10 cm): differenza ridotta a < 3%.

    STRUTTURA DEL TEST (due normalizzazioni, tutto mostrato onestamente):

      A) Normalizzazione a D(max): espone entrambe le limitazioni.
         Gamma 3%/3mm su z in [0,25]cm: pass rate atteso ~35%.
         Questo NON e un fallimento del codice: e la firma quantitativa
         dei suoi limiti fisici dichiarati.

      B) Normalizzazione a D(10 cm) [standard IAEA TRS-398]:
         Elimina l'artifact della normalizzazione diversa tra KERMA ed EGSnrc.
         Gamma 3%/3mm su z in [3,25]cm: pass rate atteso ~75%.
         Evidenzia la limitazione 1 (build-up z=3-5cm) senza nasconderla.

    CRITERIO PASS/FAIL: normalizzazione D(10 cm), zona z in [3,25]cm,
      gamma 3%/3mm, pass rate > 70%.
      La soglia 70% e giustificata dalla limitazione 1 residua a z=3-5cm.
    """

    depths_mc, pdd_mc_raw = load_pdd(PDD_WATER)
    if depths_mc is None:
        return False

    # Riferimento Sheikh-Bagheri (costanti globali REF_Z, REF_PDD)
    depths_ref = REF_Z
    pdd_ref    = REF_PDD   # normalizzato a D(max)_EGSnrc=100% a z=1.2cm

    # Normalizzazione A: D(max) — espone onestamente entrambe le limitazioni
    pdd_mc_dmax = pdd_mc_raw   # gia normalizzato a D(max)_MC=100%

    # Normalizzazione B: D(10 cm) — standard IAEA TRS-398
    iz10_mc  = int(np.argmin(np.abs(depths_mc  - 10.0)))
    iz10_ref = int(np.argmin(np.abs(depths_ref - 10.0)))
    d10_mc   = float(pdd_mc_raw[iz10_mc])
    d10_ref  = float(pdd_ref[iz10_ref])
    pdd_mc_d10  = pdd_mc_raw / d10_mc  * 100.0
    pdd_ref_d10 = pdd_ref    / d10_ref * 100.0

    DTA = 0.3   # cm = 3 mm = 1 voxel
    DD  = 3.0   # %

    def compute_gamma(pdd_mc_v, pdd_ref_v, depths_mc_v, depths_ref_v):
        mc_i = np.interp(depths_ref_v, depths_mc_v, pdd_mc_v)
        gam  = np.full(len(depths_ref_v), np.inf)
        for i in range(len(depths_ref_v)):
            dD2  = ((mc_i - pdd_ref_v[i]) / DD)**2
            dz2  = ((depths_ref_v - depths_ref_v[i]) / DTA)**2
            gam[i] = np.sqrt((dD2 + dz2).min())
        return gam, mc_i

    gamma_dmax, mc_dmax_i = compute_gamma(
        pdd_mc_dmax, pdd_ref,     depths_mc, depths_ref)
    gamma_d10,  mc_d10_i  = compute_gamma(
        pdd_mc_d10,  pdd_ref_d10, depths_mc, depths_ref)

    mask_full = (depths_ref >= 0.0) & (depths_ref <= 25.0)
    mask_post = (depths_ref >= 3.0) & (depths_ref <= 25.0)

    pr_dmax = (gamma_dmax[mask_full] <= 1.0).mean() * 100
    pr_d10  = (gamma_d10[mask_post]  <= 1.0).mean() * 100

    pass_thresh = 70.0
    passed = pr_d10 >= pass_thresh

    print(f"  ── A) Normalizzazione D(max) ────────────────────────────────────────")
    d_max_mc_z = float(depths_mc[np.argmax(pdd_mc_raw)])
    print(f"  d_max MC = {d_max_mc_z:.2f} cm  vs  d_max SB = 1.20 cm")
    print(f"  Gamma 3%/3mm, z in [0,25]cm: pass rate = {pr_dmax:.1f}%")
    print(f"  (basso per costruzione: limitazione KERMA + fascio parallelo)\n")

    print(f"  ── B) Normalizzazione D(10cm) [IAEA TRS-398] ───────────────────────")
    print(f"  D(10cm): MC={d10_mc:.2f}%  SB={d10_ref:.2f}%")
    print(f"  Gamma 3%/3mm, z in [3,25]cm: pass rate = {pr_d10:.1f}%  "
          f"(richiesto > {pass_thresh:.0f}%)\n")

    print(f"  {'z [cm]':>7}  {'MC [%]':>7}  {'SB [%]':>7}  {'Delta%':>7}  "
          f"{'gam_Dmax':>9}  {'gam_D10':>8}  Zona")
    print("  " + "─" * 70)
    for z_c in [0.15, 0.6, 1.2, 1.5, 3.0, 5.0, 10.0, 15.0, 20.0, 25.0]:
        i    = int(np.argmin(np.abs(depths_ref - z_c)))
        zona = "build-up *" if z_c < 3.0 else "post build-up"
        print(f"  {depths_ref[i]:>7.2f}  {mc_d10_i[i]:>7.2f}  "
              f"{pdd_ref_d10[i]:>7.2f}  {mc_d10_i[i]-pdd_ref_d10[i]:>+7.2f}  "
              f"{gamma_dmax[i]:>9.3f}  {gamma_d10[i]:>8.3f}  {zona}")
    print("  * differenza attesa: KERMA approximation (non errore del codice)")

    # Plot 2x2
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    ax = axes[0, 0]
    ax.plot(depths_mc, pdd_mc_dmax, 'b-', lw=2,
            label=f'MC (N={N//1_000_000:.0f}M, KERMA, fascio parallelo)')
    ax.plot(depths_ref, pdd_ref, 'ro--', ms=4, lw=1.5,
            label='Sheikh-Bagheri 2002 (EGSnrc, SAD=100cm)')
    ax.axvspan(0, 3.0, alpha=0.10, color='red', label='Build-up (KERMA != EGSnrc)')
    ax.set_ylabel('Dose / D(max) x 100 (%)', fontsize=11)
    ax.set_title('A) Normalizzazione D(max)\n[espone entrambe le limitazioni fisiche]',
                 fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylim(0, 115)

    ax = axes[0, 1]
    ax.plot(depths_mc, pdd_mc_d10, 'b-', lw=2,
            label=f'MC (N={N//1_000_000:.0f}M, norm. D10)')
    ax.plot(depths_ref, pdd_ref_d10, 'ro--', ms=4, lw=1.5,
            label='Sheikh-Bagheri 2002 (norm. D10)')
    ax.axvspan(0, 3.0, alpha=0.10, color='red', label='Build-up (Lim. 1)')
    ax.axvline(10.0, color='gray', ls=':', lw=1, alpha=0.5)
    ax.set_ylabel('Dose / D(10cm) x 100 (%)', fontsize=11)
    ax.set_title('B) Normalizzazione D(10cm) [IAEA TRS-398]\n[riduce artifact di normalizzazione]',
                 fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    z_p  = depths_ref[mask_full]
    g_p  = gamma_dmax[mask_full]
    cols = ['green' if g <= 1.0 else 'red' for g in g_p]
    ax.bar(z_p, g_p, width=0.25, color=cols, alpha=0.85)
    ax.axhline(1.0, color='black', ls='--', lw=1.5)
    ax.axvspan(0, 3.0, alpha=0.08, color='red')
    ax.set_xlabel('Profondita z (cm)', fontsize=11)
    ax.set_ylabel('Gamma index', fontsize=11)
    ax.set_title(f'Gamma 3%/3mm — norm. D(max)\npass rate = {pr_dmax:.1f}%  [atteso basso per Lim.1+2]',
                 fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    ax = axes[1, 1]
    z_p2  = depths_ref[mask_post]
    g_p2  = gamma_d10[mask_post]
    cols2 = ['green' if g <= 1.0 else 'red' for g in g_p2]
    ax.bar(z_p2, g_p2, width=0.25, color=cols2, alpha=0.85)
    ax.axhline(1.0, color='black', ls='--', lw=1.5, label='Soglia Gamma=1')
    ax.set_xlabel('Profondita z (cm)', fontsize=11)
    ax.set_ylabel('Gamma index', fontsize=11)
    ax.set_title(f'Gamma 3%/3mm — norm. D(10cm), z>=3cm\n'
                 f'pass rate = {pr_d10:.1f}%  [{PASS if passed else FAIL}]  '
                 f'(soglia {pass_thresh:.0f}%)', fontsize=10)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

    fig.suptitle(
        'TEST 10 — Confronto PDD vs Sheikh-Bagheri & Rogers (Med. Phys. 2002)\n'
        'Limitazioni dichiarate: (1) KERMA approximation  (2) fascio parallelo vs SAD=100cm',
        fontsize=11)
    plt.tight_layout()
    save_fig(fig, 'val_test10_gamma_index')

    print(f"\n  Risultato TEST 10: {PASS if passed else FAIL}")
    return passed


risultato = test10_gamma_index()  # oppure test10_gamma_index(ref_csv='mio_ref.csv')


--- VALIDAZIONE RISPETTO A LETTERATURA (6MV) ---
A) Normalizzazione al Picco (Dmax):
   Pass Rate (0-25cm): 5.4% (Basso previsto in superficie)

B) Normalizzazione a 10cm (IAEA):
   Pass Rate (3-25cm): 75.0% (Soglia: 70.0%)

  z [cm] |   MC [%] |  RIF [%] | Gamma D10
---------------------------------------------
    1.20 |   157.65 |   142.86 |    3.750
    5.00 |   128.44 |   125.00 |    1.146
   10.00 |   100.37 |   100.00 |    0.124
   20.00 |    61.67 |    63.57 |    0.634
    Figura salvata: Test_9_Validazione_pdd_letteratura.png

TEST 9 SUPERATO: True


## Test 10 - Validazione della convergenza statistica

Viene verificata la robustezza stocastica del simulatore Monte Carlo per confermare che l'incertezza statistica della dose depositata diminuisca correttamente all'aumentare del numero di storie simulate, seguendo la legge fondamentale della statistica per i processi indipendenti.

Viene eseguita una serie di simulazioni con un numero crescente di fotoni. Per ogni configurazione, la simulazione viene ripetuta più volte tramite prove indipendenti utilizzando seed casuali diversi per calcolare la varianza dei risultati. Per ogni valore di N, viene calcolata la deviazione standard della dose media depositata per fotone che rappresenta l'errore statistico intrinseco della simulazione.

In [83]:
def test_10_convergenza_statistica(salta_lento: bool = False):

    configurazioni = [(10000, 20), (100000, 20), (1000000, 20)]
    if not salta_lento:
        configurazioni.append((10000000, 5))

    print(f" ANALISI DELLA CONVERGENZA STATISTICA")
    print(f"{'N Fotoni':>12} | {'Prove':>6} | {'Media E_dep':>14} | {'Incertezza (rho)':>14}")
    print("-" * 65)

    lista_sigma = []
    lista_N = []

    for N, n_ripetizioni in configurazioni:
        risultati_energia = []

        for i in range(n_ripetizioni):
            if lancia_simulazione(N, phantom_type=0, seed=500 + i):
                dati_dose = load_dose_bin(DOSE_WATER)
                if dati_dose is not None:
                    risultati_energia.append(dati_dose.sum() / N)

        if len(risultati_energia) < 3:
            continue

        media = np.mean(risultati_energia)
        dev_standard = np.std(risultati_energia, ddof=1)

        lista_sigma.append(dev_standard)
        lista_N.append(N)

        print(f"{N:>12,} | {len(risultati_energia):>6} | {media:>14.4e} | {dev_standard:>14.4e}")
        print()

    log_N = np.log10(lista_N)
    log_sigma = np.log10(lista_sigma)
    pendenza, intercetta = np.polyfit(log_N, log_sigma, 1)

    test_superato = abs(pendenza - (-0.5)) < 0.15

    print(f"\nAnalisi della pendenza:")
    print(f"  Pendenza misurata: {pendenza:.3f}")
    print(f"  Pendenza attesa:   -0.500")
    print(f"  Risultato:         {'Coerente' if test_superato else 'Non coerente'}")
    print()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.loglog(lista_N, lista_sigma, 'bo', label='Incertezza misurata (σ)')

    N_teorico = np.logspace(np.log10(min(lista_N)), np.log10(max(lista_N)), 100)
    sigma_teorica = lista_sigma[0] * np.sqrt(lista_N[0] / N_teorico)
    ax.loglog(N_teorico, sigma_teorica, 'r-', alpha=0.7, label='Teoria 1/√N (pendenza -0.5)')

    ax.set_xlabel('Numero di particelle (N)')
    ax.set_ylabel('Incertezza (MeV/fotone)')
    ax.set_title(f'Test di Convergenza: {"Superato" if test_superato else "Fallito"}')
    ax.legend()
    ax.grid(True, which="both", alpha=0.3)

    save_fig(fig, 'Test_10_Validazione_Convergenza')

    print(f"\nTEST 10 SUPERATO: {test_superato}")
    return test_superato
risultato = test_10_convergenza_statistica()

 ANALISI DELLA CONVERGENZA STATISTICA
    N Fotoni |  Prove |    Media E_dep | Incertezza (rho)
-----------------------------------------------------------------
      10,000 |     20 |     1.0495e+00 |     9.0834e-03

     100,000 |     20 |     1.0505e+00 |     2.8167e-03

   1,000,000 |     20 |     1.0509e+00 |     1.1822e-03

  10,000,000 |      5 |     1.0508e+00 |     2.6091e-04


Analisi della pendenza:
  Pendenza misurata: -0.500
  Pendenza attesa:   -0.500
  Risultato:         OK
    Figura salvata: Test_6_Validazione_Convergenza.png

TEST 6 SUPERATO: True
